## Open in Colab

| | |
|---|---|
| **One click** | [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/getcommunityone/c1_gemma_4_good/blob/main/scripts/colab/run_in_colab.ipynb) |
| **Direct link** | https://colab.research.google.com/github/getcommunityone/c1_gemma_4_good/blob/main/scripts/colab/run_in_colab.ipynb |

**[▶ Open this notebook in Colab](https://colab.research.google.com/github/getcommunityone/c1_gemma_4_good/blob/main/scripts/colab/run_in_colab.ipynb)**

**Judges:** Colab **L4 GPU** + Secrets (`GEMINI_API_KEY`, `HF_TOKEN`) → **§0** → **§1** → **§2–§6** (one runtime, no CPU switch).

**Personal Drive:** skip §0; §1 mounts your Drive. Use **GPU** for the full run.


# 🏛️ Community One — Governance Pipeline (Gemma 4)

**Real government PDFs + audio → Gemma 4** (OCR, token budget, policy reasoning, meeting drift).

**Hackathon story hook:** *What percentage of a small town's revenue comes from speed traps?*  
Nationwide, Governing found **600+** jurisdictions where fines are **>10%** of the general fund and **284+** where they're **>20%** ([Addicted to Fines](https://www.governing.com/archive/gov-addicted-to-fines.html)). This notebook grounds that question in **real county minutes + budgets** (default: Tuscaloosa County `county_30097`).

### Quick start

1. **Runtime → L4 GPU** (or T4), **High RAM** if offered — stay on GPU the whole time.
2. Colab Secrets: **`GEMINI_API_KEY`** ([AI Studio](https://aistudio.google.com/apikey)) and **`HF_TOKEN`** ([Hugging Face](https://huggingface.co/settings/tokens)).
3. Run **§0 → §1 → §2** (`SCOPE = "fast"`) → **§3 → §4 → §5 → §6**.
4. **§7** — browse scorecard + drift analysis in the notebook output.

**Judges:** §0 downloads the public demo corpus — no personal Google Drive.


### Model ids (optional)

Resolved ids print in **§3 Install**.


### Run order

**§1** Bootstrap → **§2** `SCOPE` → **§3** Install → **§4** Keys → **§5** Inventory → **§6** pipeline (GPU).

Use **one GPU runtime** for the whole notebook. If the session restarts, re-run **§0 → §1 → §5 → §6**.

**§7+** optional reruns. Keep **`SCOPE = "fast"`** for the hackathon slot.


## § 0 — JUDGES: public corpus → local disk (no personal Google Drive)

**Judges** need API keys + public demo data — **not** your personal Google Drive.

| Keys | How |
|------|-----|
| **Cursor / local** (you have `c1_gemma_4_good/.env`) | Run §0 → §1 from repo; keys load automatically |
| **Colab in browser** | §0: **upload `.env`** *or* Colab 🔑 Secrets *or* paste keys in §0 |

1. **§0** — judge paths + load keys
2. **§1** — downloads public corpus to `/content/governance_pipeline_local` (~few min first time)
3. **§2 → §6**

**Editors:** skip §0 (uses your mounted Drive).

In [6]:
# §0 keys bootstrap: robust for local + Colab server (auto-clone if repo missing)
import os
import sys
import subprocess
from pathlib import Path
import importlib.util

_REPO_URL = "https://github.com/getcommunityone/c1_gemma_4_good.git"
_REPO_NAME = "c1_gemma_4_good"
_MARKER = Path("scripts/colab/utils/colab_secrets.py")

def _in_colab_cloud() -> bool:
    return bool(os.environ.get("COLAB_RELEASE_TAG")) and Path("/content").is_dir()

def _find_repo_root() -> Path | None:
    # 1) current dir and parents
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / _MARKER).is_file():
            return folder.resolve()

    # 2) OPEN_NAVIGATOR_ROOT if provided
    env_root = os.environ.get("OPEN_NAVIGATOR_ROOT", "").strip()
    if env_root:
        cand = Path(env_root).expanduser()
        if (cand / _MARKER).is_file():
            return cand.resolve()

    # 3) common Colab locations
    for cand in (
        Path(f"/content/{_REPO_NAME}"),
        Path("/content"),
    ):
        if (cand / _MARKER).is_file():
            return cand.resolve()

    # 4) shallow search under /content for cloned repo
    if _in_colab_cloud():
        for cand in Path("/content").glob("*/scripts/colab/utils/colab_secrets.py"):
            return cand.parents[4].resolve()  # .../<repo>/scripts/colab/utils/colab_secrets.py

    return None

def _ensure_repo_root() -> Path:
    root = _find_repo_root()
    if root is not None:
        return root

    if _in_colab_cloud():
        dest = Path(f"/content/{_REPO_NAME}")
        print(f"§0: repo not found, cloning {_REPO_URL} -> {dest}")
        rc = subprocess.call(["git", "clone", _REPO_URL, str(dest)])
        if rc != 0:
            raise RuntimeError(f"git clone failed with exit code {rc}")
        if not (dest / _MARKER).is_file():
            raise FileNotFoundError(f"Repo cloned but marker missing: {dest / _MARKER}")
        return dest.resolve()

    raise FileNotFoundError(
        "Could not find repository root with scripts/colab/utils/colab_secrets.py. "
        "Run this notebook from the c1_gemma_4_good repo root locally, "
        "or use Colab cloud where §0 can auto-clone the repo."
    )

def _load_colab_secrets_module(repo_root: Path):
    secrets_py = repo_root / _MARKER
    if not secrets_py.is_file():
        raise FileNotFoundError(f"Missing colab_secrets.py at {secrets_py}")

    utils_dir = repo_root / "scripts" / "colab" / "utils"
    for p in (repo_root, utils_dir):
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))

    try:
        import scripts.colab.utils.colab_secrets as mod
        return mod
    except Exception:
        spec = importlib.util.spec_from_file_location("colab_secrets", str(secrets_py))
        mod = importlib.util.module_from_spec(spec)
        assert spec.loader is not None
        spec.loader.exec_module(mod)
        return mod

repo_root = _ensure_repo_root()
os.environ["OPEN_NAVIGATOR_ROOT"] = str(repo_root)
mod = _load_colab_secrets_module(repo_root)

# Stop Colab Secrets UI timeouts in local/extension runs, but allow real Colab to use notebook secrets.
if hasattr(mod, "in_colab_runtime") and mod.in_colab_runtime():
    os.environ.setdefault("GOVERNANCE_COLAB_SINGLE_RUNTIME", "1")
    os.environ.setdefault("GOVERNANCE_COLAB_TWO_PHASE", "0")
    os.environ.setdefault("GOVERNANCE_COLAB_SKIP_CONFIRM_UI", "1")
    os.environ.pop("GOVERNANCE_NOTEBOOK_SECRETS", None)
    os.environ.pop("GOVERNANCE_SKIP_COLAB_SECRETS", None)
elif hasattr(mod, "prefer_env_over_colab_secrets"):
    mod.prefer_env_over_colab_secrets()
else:
    os.environ["GOVERNANCE_NOTEBOOK_SECRETS"] = "env_only"

# Optional: load .env keys immediately if helper exists.
if hasattr(mod, "load_api_keys_to_environ"):
    mod.load_api_keys_to_environ(repo=repo_root)

# Colab fallback when Secrets time out: paste key here OR set GEMINI_API_KEY_INLINE in env
_inline_gemini = os.environ.get("GEMINI_API_KEY_INLINE", "").strip()
if _inline_gemini and not (os.environ.get("GEMINI_API_KEY") or "").strip():
    os.environ["GEMINI_API_KEY"] = _inline_gemini
    print("§0: GEMINI_API_KEY from GEMINI_API_KEY_INLINE")

print("§0 ready:", repo_root)
print("GOVERNANCE_NOTEBOOK_SECRETS=", os.environ.get("GOVERNANCE_NOTEBOOK_SECRETS", ""))
print("GEMINI_API_KEY set:", bool((os.environ.get("GEMINI_API_KEY") or "").strip()))

§0 ready: /content/c1_gemma_4_good
GOVERNANCE_NOTEBOOK_SECRETS= 


In [ ]:
# Judge mode guardrail for Colab: use a safe default shared-folder URL and auto-correct broad parent URLs
import os
import re
from pathlib import Path

_IN_COLAB_CLOUD = bool(os.environ.get("COLAB_RELEASE_TAG")) and Path("/content").is_dir()
_SAFE_DEFAULT_FOLDER_URL = "https://drive.google.com/drive/folders/1eTADXbP80aewDiG-sV-qCiD2UhIFbkno?usp=sharing"
_DISALLOWED_PARENT_FOLDER_IDS = {
    "1H_narmvkEUEalAyvl1P2oY7XbzaVMD7_",  # broad parent that pulls full hackathons tree
}


def _drive_folder_id(value: str) -> str:
    if not value:
        return ""
    m = re.search(r"/folders/([a-zA-Z0-9_-]+)", value)
    if m:
        return m.group(1)
    if re.fullmatch(r"[a-zA-Z0-9_-]{20,}", value.strip()):
        return value.strip()
    return ""


if _IN_COLAB_CLOUD:
    _raw_url = (os.environ.get("GOVERNANCE_RAW_INPUTS_DRIVE_FOLDER_URL") or "").strip()
    if not _raw_url:
        _raw_url = _SAFE_DEFAULT_FOLDER_URL
        os.environ["GOVERNANCE_RAW_INPUTS_DRIVE_FOLDER_URL"] = _raw_url
        print("No shared folder URL provided; using safe default communityone_public URL.")

    _folder_id = _drive_folder_id(_raw_url)
    if _folder_id in _DISALLOWED_PARENT_FOLDER_IDS:
        print("Detected disallowed parent folder ID. Auto-correcting to safe default URL.")
        _raw_url = _SAFE_DEFAULT_FOLDER_URL
        os.environ["GOVERNANCE_RAW_INPUTS_DRIVE_FOLDER_URL"] = _raw_url
        _folder_id = _drive_folder_id(_raw_url)  # Re-extract folder ID after correction
        print("Corrected URL:", _raw_url)
        print("Re-extracted folder ID:", _folder_id)

    if _folder_id in _DISALLOWED_PARENT_FOLDER_IDS:
        raise RuntimeError(
            "GOVERNANCE_RAW_INPUTS_DRIVE_FOLDER_URL points to a broad parent folder.\n"
            "Use the exact communityone_public folder URL, then re-run Section 0 and Section 1."
        )

    os.environ["GOVERNANCE_JUDGE_MODE"] = "1"
    os.environ.setdefault("GOVERNANCE_PIPELINE_DATA_ROOT", "/content/governance_pipeline_local")
    os.environ.setdefault("GOVERNANCE_COLAB_SINGLE_RUNTIME", "1")
    os.environ.setdefault("GOVERNANCE_COLAB_SKIP_CONFIRM_UI", "1")
    os.environ.setdefault("GOVERNANCE_COLAB_SKIP_GPU_CONFIRM", "1")
    os.environ.setdefault("OPEN_NAVIGATOR_ROOT", "/content/c1_gemma_4_good")
    os.environ.pop("GOVERNANCE_NOTEBOOK_SECRETS", None)
    os.environ.pop("GOVERNANCE_SKIP_COLAB_SECRETS", None)
    print("Judge mode ON (shared folder URL ready).")
    print("Judge path: **L4 GPU** for the whole run → §6 pipeline cell (no CPU switch).")
    print("Shared folder:", _raw_url)
    print("Pipeline root:", os.environ["GOVERNANCE_PIPELINE_DATA_ROOT"])
else:
    print("Local runtime detected; no Colab Drive mount path changes applied.")

Judge mode ON (shared folder URL ready).
Shared folder: https://drive.google.com/drive/folders/1eTADXbP80aewDiG-sV-qCiD2UhIFbkno?usp=sharing
Pipeline root: /content/governance_pipeline_local


In [8]:
# Local setup: configure pipeline data root for non-Colab environments
import os
from pathlib import Path

# If not in Colab and no data root is set, create a local one
if not (os.environ.get("COLAB_RELEASE_TAG") and Path("/content").is_dir()):
    # Not in Colab — set up local pipeline root
    local_root = Path("/tmp/governance_pipeline_local")
    local_root.mkdir(parents=True, exist_ok=True)
    os.environ["GOVERNANCE_PIPELINE_DATA_ROOT"] = str(local_root)
    os.environ.setdefault("GOVERNANCE_NOTEBOOK_SECRETS", "env_only")
    print(f"Local setup: GOVERNANCE_PIPELINE_DATA_ROOT = {local_root}")


In [15]:
# §1 Bootstrap — repo + Drive + hackathon data root
# (Re-run after every Runtime → Restart session.)

import os
import sys
from pathlib import Path

_REPO_MARKER = Path("scripts/colab/utils/colab_paths.py")
_IN_COLAB_CLOUD = bool(os.environ.get("COLAB_RELEASE_TAG")) and Path("/content").is_dir()

# Local/extension runs skip Colab userdata to avoid UI timeouts; real Colab may use notebook secrets.
if not _IN_COLAB_CLOUD:
    os.environ.setdefault("GOVERNANCE_NOTEBOOK_SECRETS", "env_only")
else:
    os.environ.pop("GOVERNANCE_NOTEBOOK_SECRETS", None)
    os.environ.pop("GOVERNANCE_SKIP_COLAB_SECRETS", None)


def _repo_ok(root: Path) -> bool:
    return (root / _REPO_MARKER).is_file()


def _load_dotenv_file(dot: Path) -> bool:
    if not dot.is_file():
        return False
    for _line in dot.read_text(encoding="utf-8").splitlines():
        _line = _line.strip()
        if not _line or _line.startswith("#") or "=" not in _line:
            continue
        _k, _, _v = _line.partition("=")
        _k, _v = _k.strip(), _v.strip().strip('"').strip("'")
        if _k and (_k not in os.environ or not os.environ[_k].strip()):
            os.environ[_k] = _v
    print(f"§1: loaded {dot}")
    return True


for _folder in (Path.cwd(), *Path.cwd().parents):
    if _load_dotenv_file(_folder / ".env"):
        break


def _find_or_clone_repo() -> Path:
    env = os.environ.get("OPEN_NAVIGATOR_ROOT", "").strip()
    if env:
        cand = Path(env).expanduser()
        if _repo_ok(cand):
            return cand.resolve()
        if _IN_COLAB_CLOUD:
            print(
                f"§1: ignoring OPEN_NAVIGATOR_ROOT={env!r} "
                "(local path — not on this Colab VM)"
            )

    for folder in (Path("/content/c1_gemma_4_good"), Path.cwd(), *Path.cwd().parents):
        if _repo_ok(folder):
            return folder.resolve()

    dest = Path("/content/c1_gemma_4_good")
    print("Downloading c1_gemma_4_good from GitHub (first run this Colab session)…")
    rc = os.system(
        "git clone https://github.com/getcommunityone/c1_gemma_4_good.git "
        f"{dest}"
    )
    if rc != 0:
        raise RuntimeError(f"git clone failed (exit {rc})")
    if not _repo_ok(dest):
        raise RuntimeError(
            "Clone finished but colab_paths.py is missing. "
            "Open run_in_colab.ipynb from GitHub and re-run §1."
        )
    return dest.resolve()


REPO_PATH = _find_or_clone_repo()
os.environ["OPEN_NAVIGATOR_ROOT"] = str(REPO_PATH)
for _entry in (
    str(REPO_PATH),
    str(REPO_PATH / "scripts/colab"),
    str(REPO_PATH / "scripts/colab/utils"),
):
    if _entry not in sys.path:
        sys.path.insert(0, _entry)

from colab_bootstrap import complete_section1_bootstrap

REPO_PATH, PATHS = complete_section1_bootstrap(REPO_PATH)

# Keys: laptop .env does NOT travel to Colab — load Drive .env, clone .env, or Secrets.
if not (os.environ.get("GEMINI_API_KEY") or "").strip():
    try:
        from colab_secrets import get_notebook_secret

        _g = get_notebook_secret("GEMINI_API_KEY")
        if _g:
            os.environ["GEMINI_API_KEY"] = _g
            print("§1: GEMINI_API_KEY from Colab Secrets")
        _h = get_notebook_secret("HF_TOKEN")
        if _h:
            os.environ["HF_TOKEN"] = _h
    except Exception:
        pass

_has_gemini = bool((os.environ.get("GEMINI_API_KEY") or "").strip())
if _has_gemini:
    print("§1 keys: GEMINI_API_KEY is set in this kernel.")
elif _IN_COLAB_CLOUD:
    print(
        "\n§1 keys: GEMINI_API_KEY still missing on this Colab VM.\n"
        "  • Judges: Colab 🔑 Secrets → GEMINI_API_KEY (notebook access ON)\n"
        "  • Laptop .env is NOT on Colab — only API keys via Secrets or upload\n"
        f"  • Optional: {REPO_PATH / '.env'}\n"
    )


Updating c1_gemma_4_good from GitHub (main)…
Judge corpus: reuse local copy at /content/governance_pipeline_local/01_raw_inputs
§1 judge sync: pipeline=/content/governance_pipeline_local  raw_inputs=/content/governance_pipeline_local/01_raw_inputs
§1: no API keys on this Colab VM yet.
      • Judges: Colab 🔑 Secrets → GEMINI_API_KEY (notebook access ON)
      (Laptop .env is not on Colab; corpus data is separate from API keys.)

§1 Bootstrap OK — you can continue to §2
Environment:     Google Colab
Code (repo):     /content/c1_gemma_4_good
Hackathon data:  /content/governance_pipeline_local

Judge mode: public corpus on local disk (no personal Drive required).
  Raw inputs: /content/governance_pipeline_local/01_raw_inputs
   ⚠ Colab secret 'GEMINI_API_KEY' unavailable (TimeoutException: Requesting secret GEMINI_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.). Falling back to env / .env.
   ⚠ Colab secret 'GEMINI_API_KEY' unavailable (TimeoutException: Re

### API keys on **Google Colab cloud**

The git clone at `/content/c1_gemma_4_good` has **no** `.env` (not in GitHub). Use **one** of:

1. **Colab 🔑 Secrets** — `GEMINI_API_KEY` and `HF_TOKEN` (notebook access **ON**), or
2. **Copy `.env` to the clone** — e.g. upload to `/content/c1_gemma_4_good/.env`, or
3. **`.env` on Drive** — e.g. `My Drive/CommunityOne/hackathons/2026_Gemma_4_Good/.env` (§1 loads it after Drive mount).

### Optional — Cursor / local Jupyter only

Repo `.env` at project root; set `OPEN_NAVIGATOR_ROOT` if §1 cannot find the checkout.


## 2 · Demo scope

Edit **`SCOPE`** in the next cell. **`MEDIA_SCOPE`** in §2 is optional; **§6** runs PDF + video together on GPU.

| `SCOPE` | Time | States | Jurisdictions | Meeting dates | PDFs / jur | Audio chunks | Contact images |
|--------|------|--------|---------------|---------------|------------|--------------|----------------|
| **`fast`** (default) | ~45–75 min (2 phases) | 1 | 1 (`county_30097`) | **2026-02-18 only** | **6** | **2 recordings** × 4 chunks | off |

**`fast`** = **PDF + video** across two runtimes, pinned **2026-02-18**, Tuscaloosa County.

| **`medium`** | ~35–50 min | 2 | 2 | 2 | 2 | 3 | 6 |
| **`full`** | ~90+ min | 4 | 4 | 3 | 3 | 4 | 12 |

Override: `os.environ["GOVERNANCE_DEMO_JURISDICTION_SLUG"] = "county_30097"`.


In [10]:
# ── Edit these lines only ───────────────────────────────────────
SCOPE = "fast"  # default: Tuscaloosa county_30097, 2026-02-18, PDF+video (all)
# MEDIA_SCOPE: None = use SCOPE preset (fast → "all"); or override: pdf | audio | video
MEDIA_SCOPE = None
# ─────────────────────────────────────────────────────────────────

In [11]:
import os
import sys

# Repo path is fixed in cell 4 (git fetch + reset --hard ran there).
proj = PATHS.project_path

assert (proj / "prompts" / "policy_analysis_v1.md").is_file(), (
    f"Missing {proj}/prompts/policy_analysis_v1.md"
)

sys.path.insert(0, str(proj))
COLAB_DIR = proj / "scripts" / "colab"
sys.path.insert(0, str(COLAB_DIR))

# Drop stale Colab copies of scripts/colab/*.py so imports pick up a fresh git pull.
for _mod in list(sys.modules):
    if _mod in (
        "gatekeeper_triage",
        "governance_meeting_llm",
        "gemma_hf_backend",
        "demo_scope",
        "pipeline_media_scope",
    ):
        del sys.modules[_mod]

# Pin the resolved Drive path so downstream modules read it from the env.
os.environ.setdefault("GOVERNANCE_PIPELINE_DATA_ROOT", str(PATHS.governance_pipeline_data))

from scripts.utils.gdrive_paths import GovernancePipelinePaths

PIPE = GovernancePipelinePaths.resolve()
# Create only the subfolders we will write into (03_processed_outputs/*). We do
# NOT create 01_raw_inputs here — if the resolver pointed somewhere wrong, an
# empty 01_raw_inputs would mask the bug.
for _sub in (PIPE.transcripts, PIPE.gemma_json, PIPE.human_summaries):
    _sub.mkdir(parents=True, exist_ok=True)

from governance_meeting_llm import (
    AUDIO_EXTS,
    PDF_EXTS,
    TOKEN_BUDGET_HIGH,
    TOKEN_BUDGET_LOW,
    call_google_genai_multimodal,
    transcribe_audio_with_gemma,
    TRANSCRIPTION_SUPPORTED_LANGUAGES,
    chunk_audio_ffmpeg,
    demo4_drift_output_complete,
    find_existing_audio_chunks,
    force_reprocess_outputs,
    policy_chunk_output_complete,
    read_json_file,
    text_output_complete,
    classify_pdf_page_heuristic,
    extract_pdf_digital_text,
    is_scanned_pdf,
    load_contacts_lookup,
    load_meeting_data_lookup,
    load_text_file,
    merge_contacts_by_jurisdiction,
    merge_meeting_data_by_jurisdiction,
    mime_for,
    mirror_output_path,
    parse_policy_analysis_response,
    policy_drift_summarize,
    render_pdf_pages,
    select_demo4_media,
    walk_raw_inputs,
    embed_text_with_gemma,
    cosine_similarity_matrix,
    shield_review_text,
)

PROMPT_PATH = proj / "prompts" / "policy_analysis_v1.md"
POLICY_PROMPT = load_text_file(PROMPT_PATH)
print("Loaded prompt chars:", len(POLICY_PROMPT))
print("Pipeline data root:", PIPE.root)


Loaded prompt chars: 46206
Pipeline data root: /content/governance_pipeline_local


In [12]:
# § Validate raw inputs resolution (shared-folder mode)
from scripts.utils.gdrive_paths import resolve_governance_raw_inputs_root

print("─" * 70)
print("§ RAW INPUTS RESOLUTION CHECK")
print("─" * 70)
print(f"Pipeline root: {os.environ.get('GOVERNANCE_PIPELINE_DATA_ROOT', '(using default)')}")
print(f"Raw inputs override: {os.environ.get('GOVERNANCE_RAW_INPUTS_ROOT', '(not set)')}")
print(f"Shared folder URL: {os.environ.get('GOVERNANCE_RAW_INPUTS_DRIVE_FOLDER_URL', '(not set)')}")
print()

try:
    RAW_ROOT = resolve_governance_raw_inputs_root(PIPE.root)
    print(f"✓ Resolved RAW_ROOT: {RAW_ROOT}")
    
    if RAW_ROOT.is_dir():
        inv_dirs = list(RAW_ROOT.glob("*/"))
        print(f"✓ RAW_ROOT exists and is readable ({len(inv_dirs)} top-level dirs)")
        for d in sorted(inv_dirs)[:5]:
            print(f"  • {d.name}/")
        if len(inv_dirs) > 5:
            print(f"  … +{len(inv_dirs) - 5} more")
    else:
        print(f"⚠ RAW_ROOT path exists but is not a directory: {RAW_ROOT}")
except Exception as e:
    print(f"✗ Failed to resolve RAW_ROOT: {e}")

print()
print("GATEKEEPER LOGS:")
from scripts.colab.meetings.jurisdiction_pipeline import gatekeeper_logs_dir
logs_dir = gatekeeper_logs_dir(PIPE.root)
print(f"  → {logs_dir}")
print()
print("─ JUDGE MODE: How shared folders are accessed ─")
print("  • Mounted path: checks .shortcut-targets-by-id/<id> if Drive folder URL is set")
print("  • Drive API: In Colab, automatically downloads from shared folder")
print("    (safe, no broad permissions, no Google Drive for Desktop needed)")
print("  • Local WSL: resolves via mounted Drive path (requires Google Drive Desktop)")
print("─" * 70)


──────────────────────────────────────────────────────────────────────
§ RAW INPUTS RESOLUTION CHECK
──────────────────────────────────────────────────────────────────────
Pipeline root: /content/governance_pipeline_local
Raw inputs override: /content/governance_pipeline_local/01_raw_inputs
Shared folder URL: https://drive.google.com/drive/folders/1eTADXbP80aewDiG-sV-qCiD2UhIFbkno?usp=sharing

✓ Resolved RAW_ROOT: /content/governance_pipeline_local/01_raw_inputs
✓ RAW_ROOT exists and is readable (1 top-level dirs)
  • 2026_Gemma_4_Good/

GATEKEEPER LOGS:
  → /content/governance_pipeline_local/00_logs

─ JUDGE MODE: How shared folders are accessed ─
  • Mounted path: checks .shortcut-targets-by-id/<id> if Drive folder URL is set
  • Drive API: In Colab, automatically downloads from shared folder
    (safe, no broad permissions, no Google Drive for Desktop needed)
  • Local WSL: resolves via mounted Drive path (requires Google Drive Desktop)
──────────────────────────────────────────────

## 3 · Install dependencies

Lightweight install (**L4 GPU** recommended for §6 — Gatekeeper + PDF use Google AI Studio; video uses GPU + `HF_TOKEN`).

**Gatekeeper defaults to Google AI Studio** (no ~10GB model download). `transformers` is installed in **§4** only when you enable Hugging Face (`GOVERNANCE_GATEKEEPER_FORCE_HF=1` or `GOVERNANCE_LLM_BACKEND=huggingface`).


In [13]:
# poppler-utils for pdf2image (Gatekeeper). Skipped off Colab.
import shutil, subprocess, sys
if not shutil.which("pdftoppm"):
    try:
        subprocess.run(["apt-get", "-qq", "install", "-y", "poppler-utils"], check=False)
    except FileNotFoundError:
        pass

subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "google-genai>=1.0", "librosa", "pymupdf", "pdf2image", "ipywidgets"
    ]
)
print(
    "Default: Google AI Studio for PDF; L4 GPU + HF_TOKEN for video. "
    "If GOVERNANCE_GATEKEEPER_FORCE_HF=1, §4 installs transformers>=5.5.0 and loads E2B on GPU."
)

Default: Google AI Studio (CPU runtime OK). If GOVERNANCE_GATEKEEPER_FORCE_HF=1, §4 installs transformers>=5.5.0 and loads E2B on GPU.


## 4 · API keys and models

Uses caps from **§2** (`SCOPE`). Prints resolved Gemma ids for your API key.

**Where keys come from** (first match wins): repo **`.env`** → **`os.environ`** → Colab **Secrets** (🔑).  
If Colab Secrets time out, §4 still runs when keys are in **`.env`** or you set `GOVERNANCE_NOTEBOOK_SECRETS=env_only`.

- **§6 (GPU):** `GEMINI_API_KEY` and `HF_TOKEN` in Colab Secrets (or `.env` locally).
- **Local / Cursor:** copy `.env.example` → `.env`, fill keys, re-run **§1** then **§4**.


In [14]:
import os
import sys
import importlib.util
from pathlib import Path

def _load_dotenv_at(repo: Path) -> bool:
    dot = repo / ".env"
    if not dot.is_file():
        return False
    for line in dot.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        k, v = k.strip(), v.strip().strip('"').strip("'")
        if k and (k not in os.environ or not os.environ[k].strip()):
            os.environ[k] = v
    return True

def _resolve_repo_for_keys() -> Path:
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "scripts/colab/utils/colab_paths.py").is_file():
            return folder.resolve()
    env = os.environ.get("OPEN_NAVIGATOR_ROOT", "").strip()
    if env and (Path(env) / "scripts/colab/utils/colab_paths.py").is_file():
        return Path(env).expanduser().resolve()
    if "proj" in globals():
        return Path(proj).resolve()
    if "PATHS" in globals():
        return Path(PATHS.project_path).resolve()
    if "REPO_PATH" in globals():
        return Path(REPO_PATH).resolve()
    return Path.cwd().resolve()

_repo_for_env = _resolve_repo_for_keys()
_colab_utils = _repo_for_env / "scripts" / "colab" / "utils"
for _p in (_repo_for_env, _repo_for_env / "scripts" / "colab", _colab_utils):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
try:
    from colab_secrets import (
        default_local_secrets_mode,
        get_notebook_secret,
        in_colab_runtime,
        load_dotenv_all_candidates,
        notebook_runs_locally,
        sanitize_api_key as _sanitize_key,
    )
except ModuleNotFoundError:
    _secrets_py = _colab_utils / "colab_secrets.py"
    _spec = importlib.util.spec_from_file_location("colab_secrets", _secrets_py)
    _mod = importlib.util.module_from_spec(_spec)
    assert _spec.loader is not None
    _spec.loader.exec_module(_mod)
    default_local_secrets_mode = _mod.default_local_secrets_mode
    get_notebook_secret = _mod.get_notebook_secret
    in_colab_runtime = _mod.in_colab_runtime
    load_dotenv_all_candidates = _mod.load_dotenv_all_candidates
    notebook_runs_locally = _mod.notebook_runs_locally
    _sanitize_key = _mod.sanitize_api_key

def _get_colab_secret_direct(name: str):
    if not in_colab_runtime():
        return None
    try:
        from google.colab import userdata
    except Exception as exc:
        print(f"§4: google.colab.userdata unavailable for {name}: {type(exc).__name__}: {exc}")
        return None
    try:
        value = (userdata.get(name) or "").strip()
    except Exception as exc:
        print(f"§4: Colab secret lookup failed for {name}: {type(exc).__name__}: {exc}")
        return None
    if value:
        print(f"§4: loaded {name} from Colab Secrets")
    return value or None

if in_colab_runtime():
    os.environ.pop("GOVERNANCE_NOTEBOOK_SECRETS", None)
    os.environ.pop("GOVERNANCE_SKIP_COLAB_SECRETS", None)
    print("§4: Colab detected — forcing notebook Colab Secrets first for API keys.")
default_local_secrets_mode()
_pipe_root = PATHS.governance_pipeline_data if "PATHS" in globals() else None
_env_dir = load_dotenv_all_candidates(repo=_repo_for_env, pipeline_root=_pipe_root)
if _env_dir:
    print(f"§4: loaded {_env_dir / '.env'}")
elif notebook_runs_locally():
    print("Keys: local → .env (not Colab Secrets UI)")
elif in_colab_runtime():
    print("§4: no .env yet — trying Colab 🔑 Secrets for GEMINI_API_KEY / HF_TOKEN first")

if "SCOPE" not in globals():
    SCOPE = "fast"
if "MEDIA_SCOPE" not in globals():
    MEDIA_SCOPE = None

for _scope_path in (
    _repo_for_env / "scripts" / "colab",
    _repo_for_env / "scripts" / "colab" / "demos",
    _repo_for_env / "scripts" / "colab" / "media",
):
    if str(_scope_path) not in sys.path:
        sys.path.insert(0, str(_scope_path))

from demo_scope import apply_scope_and_media

ACTIVE_PRESET, _active_media_key, ACTIVE_MEDIA = apply_scope_and_media(SCOPE, MEDIA_SCOPE)

if str(proj / "scripts" / "colab") not in sys.path:
    sys.path.insert(0, str(proj / "scripts" / "colab"))
_colab_utils = proj / "scripts" / "colab" / "utils"
if str(_colab_utils) not in sys.path:
    sys.path.insert(0, str(_colab_utils))
# Judge-friendly defaults (edit here only if needed)
os.environ.setdefault("GOVERNANCE_MODE", "DEMO")
os.environ.setdefault("GOVERNANCE_LLM_BACKEND", "google")
os.environ.setdefault("GOVERNANCE_LOG_LLM_CATALOG", "0")
os.environ.setdefault("GOVERNANCE_ORGANIZE_MEETINGS", "1")
os.environ.setdefault("GOVERNANCE_ORGANIZE_INCLUDE_FLAT_PDFS", "1")
os.environ.setdefault("GOVERNANCE_RUN_DEMO3_WITH_VIDEO", "1")
os.environ.setdefault("GOVERNANCE_DEMO_DATE_SCOPE", "1")
os.environ.setdefault("GOVERNANCE_DEMO_YEAR_SCOPE", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_PDF_DPI", "120")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_COPY_TO_TMP", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_LARGE_PDF_BYTES", "1500000")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_TEXT_FIRST", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_TEXT_MIN_CHARS", "80")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_FSYNC", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_API_TIMEOUT_SECONDS", "60")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_PDF_DIRECT", "0")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_PDF_PAGES", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_SKIP_PDF_PROBE", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_MAX_MEETING_SESSIONS", "3")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_MAX_FILES", "12")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_RULES_ONLY", "1")
# Demo 4: Hugging Face E2B/E4B (native audio/video); Opus slices cached on Drive.
os.environ.setdefault("GOVERNANCE_DEMO4_USE_HF", "1")
os.environ.setdefault("GOVERNANCE_DEMO4_HF_MODEL", "google/gemma-4-E2B-it")
os.environ.setdefault("GOVERNANCE_TRANSCRIPTION_HF_MODEL", "google/gemma-4-E2B-it")
os.environ.setdefault("GOVERNANCE_DRIFT_USE_HF", "0")
os.environ.setdefault("GOVERNANCE_DEMO4_MODEL", "")
os.environ.setdefault("GOVERNANCE_DEMO4_PREFER_OPUS", "1")
# Progress: wall-clock timestamps, step timing, heartbeat while HF/ffmpeg/generate run
os.environ.setdefault("GOVERNANCE_STEP_TIMESTAMPS", "1")
os.environ.setdefault("GOVERNANCE_STEP_TIMING", "1")
os.environ.setdefault("GOVERNANCE_HEARTBEAT_SECONDS", "45")
os.environ.setdefault("GOVERNANCE_DEMO3_API_TIMEOUT_SECONDS", "900")
# Demo 3: digital text when present; light OCR for scans; policy on text (not full-PDF vision)
os.environ.setdefault("GOVERNANCE_DEMO3_INPUT", "auto")
os.environ.setdefault("GOVERNANCE_DEMO_THINKING_BUDGET", "0")
os.environ.setdefault("GOVERNANCE_DEMO3_OCR_RESOLUTION", "LOW")
os.environ.setdefault("GOVERNANCE_DEMO3_MAX_DOCUMENT_CHARS", "60000")
os.environ.setdefault("GOVERNANCE_DEMO3_AGENDA_MODE", "skip")
os.environ.setdefault("GOVERNANCE_DEMO3_AGENDA_MAX_OUTPUT_TOKENS", "2048")
os.environ.setdefault("GOVERNANCE_DEMO3_AGENDA_API_TIMEOUT_SECONDS", "180")
os.environ.setdefault("GOVERNANCE_COLAB_SKIP_CONFIRM_UI", "0")
os.environ.setdefault("GOVERNANCE_DEMO1_SKIP_WHEN_DIGITAL", "1")
os.environ.setdefault("GOVERNANCE_DEMO2_SKIP_WHEN_DIGITAL", "1")
os.environ.setdefault("GOVERNANCE_COLAB_SINGLE_RUNTIME", "1")
os.environ.setdefault("GOVERNANCE_COLAB_TWO_PHASE", "0")
os.environ.setdefault("GOVERNANCE_COLAB_SKIP_CONFIRM_UI", "1")
os.environ.setdefault("GOVERNANCE_PIPELINE_VERBOSE", "1")
os.environ.setdefault("GOVERNANCE_PIPELINE_PROGRESS", "1")
# Demo 4 video/mp4 chunks: set by MEDIA_SCOPE in §2 (video → 1, audio → 0, all → 1)
os.environ.setdefault("GOVERNANCE_DEMO4_VIDEO_CHUNKS", "1")
# os.environ.setdefault("GOVERNANCE_GATEKEEPER_FORCE_HF", "1")  # opt-in: ~10GB HF download

LLM_BACKEND = os.environ.get("GOVERNANCE_LLM_BACKEND", "google").strip().lower()
USE_HF_ALL = LLM_BACKEND in ("huggingface", "hf", "local")

_hf_secret = _get_colab_secret_direct("HF_TOKEN")
if _hf_secret:
    os.environ["HF_TOKEN"] = _hf_secret
HF_TOKEN = _sanitize_key(
    _hf_secret
    or get_notebook_secret("HF_TOKEN", repo=_repo_for_env)
    or os.environ.get("HF_TOKEN")
)

_gemini_secret = _get_colab_secret_direct("GEMINI_API_KEY")
_google_secret = _get_colab_secret_direct("GOOGLE_API_KEY") if not _gemini_secret else None
if _gemini_secret:
    os.environ["GEMINI_API_KEY"] = _gemini_secret
elif _google_secret:
    os.environ["GOOGLE_API_KEY"] = _google_secret
GEMINI_API_KEY = _sanitize_key(
    _gemini_secret
    or _google_secret
    or get_notebook_secret("GEMINI_API_KEY", repo=_repo_for_env)
    or get_notebook_secret("GOOGLE_API_KEY", repo=_repo_for_env)
    or os.environ.get("GEMINI_API_KEY")
    or os.environ.get("GOOGLE_API_KEY")
)

if not GEMINI_API_KEY:
    _hint = (
        "Colab: Colab 🔑 Secrets → GEMINI_API_KEY (notebook access ON), "
        f"OR copy .env to {_repo_for_env / '.env'}, "
        "OR put .env in your hackathon folder on Google Drive."
        if in_colab_runtime()
        else f"Put GEMINI_API_KEY in {_repo_for_env / '.env'} then re-run §1 and §4."
    )
    raise RuntimeError(f"GEMINI_API_KEY missing. {_hint}")
if not GEMINI_API_KEY.startswith("AIza") or len(GEMINI_API_KEY) < 35:
    raise RuntimeError(f"GEMINI_API_KEY looks malformed (len={len(GEMINI_API_KEY)}).")
API_KEY = GEMINI_API_KEY

GENAI_MODEL = os.environ.get("GOVERNANCE_GENAI_MODEL", "gemma-4-26b-a4b-it").strip()
THINKING_MODEL = os.environ.get("GOVERNANCE_THINKING_MODEL", "gemma-4-31b-it").strip()
GATEKEEPER_MODEL = os.environ.get("GOVERNANCE_GATEKEEPER_MODEL", "gemma-4-e2b-it").strip()
EMBEDDING_MODEL = os.environ.get("GOVERNANCE_EMBEDDING_MODEL", "embeddinggemma-300m").strip()
SHIELD_MODEL = os.environ.get("GOVERNANCE_SHIELD_MODEL", "shieldgemma-9b").strip() or GENAI_MODEL

from gemma_hf_backend import (
    gatekeeper_use_huggingface,
    model_requires_huggingface,
    preload_gemma_hf,
    print_hf_model_catalog,
    resolve_hf_model_id,
    ensure_transformers_gemma4_ready,
    _HF_GATEKEEPER_REPO_DEFAULT,
)

NEEDS_HF = (
    USE_HF_ALL
    or model_requires_huggingface(GENAI_MODEL)
    or gatekeeper_use_huggingface()
)
if NEEDS_HF:
    if not HF_TOKEN:
        raise RuntimeError(
            "HF_TOKEN required when GOVERNANCE_GATEKEEPER_FORCE_HF=1 or GOVERNANCE_LLM_BACKEND=huggingface. "
            "Default Gatekeeper uses AI Studio only — no HF_TOKEN needed."
        )
    os.environ["HF_TOKEN"] = HF_TOKEN

_log_llm_catalog = os.environ.get("GOVERNANCE_LOG_LLM_CATALOG", "0") == "1"

from gatekeeper_triage import (
    _GEMMA_HEAVY_FALLBACKS,
    _GEMMA_THINKING_FALLBACKS,
    _GEMMA_DEMO4_AUDIO_FALLBACKS,
    _GEMMA_GATEKEEPER_AI_FALLBACKS,
    _GEMMA_TRIAGE_FALLBACKS,
    _build_genai_client,
    print_available_models,
    resolve_model_id,
)

_resolver_client = _build_genai_client(GEMINI_API_KEY)
if _log_llm_catalog:
    print_available_models(
        _resolver_client,
        requested=(GENAI_MODEL, GATEKEEPER_MODEL, EMBEDDING_MODEL, SHIELD_MODEL),
        role="Google AI Studio",
    )

if model_requires_huggingface(GENAI_MODEL) or model_requires_huggingface(THINKING_MODEL):
    GENAI_MODEL = resolve_hf_model_id(GENAI_MODEL)
    THINKING_MODEL = resolve_hf_model_id(THINKING_MODEL)
    ensure_transformers_gemma4_ready(auto_install=True)
    if _log_llm_catalog:
        print_hf_model_catalog(requested=(GENAI_MODEL, THINKING_MODEL), role="Hugging Face (demos)")
    print(f"Loading HF weights for demos: {GENAI_MODEL} …")
    preload_gemma_hf(GENAI_MODEL, load_audio_variant=True)
    if THINKING_MODEL != GENAI_MODEL:
        print(f"Loading HF weights for Demo 3 thinking: {THINKING_MODEL} …")
        preload_gemma_hf(THINKING_MODEL, load_audio_variant=True)
else:
    GENAI_MODEL = resolve_model_id(
        _resolver_client, GENAI_MODEL, fallbacks=_GEMMA_HEAVY_FALLBACKS, role="Demos 1/2 PDF (AI Studio)"
    )
    THINKING_MODEL = resolve_model_id(
        _resolver_client,
        THINKING_MODEL,
        fallbacks=_GEMMA_THINKING_FALLBACKS,
        role="Demo 3 thinking (AI Studio)",
    )
if gatekeeper_use_huggingface():
    ensure_transformers_gemma4_ready(auto_install=True)
    hf_gate = (
        os.environ.get("GOVERNANCE_GATEKEEPER_MODEL", "").strip()
        or os.environ.get("GOVERNANCE_HF_GATEKEEPER_MODEL", "").strip()
        or _HF_GATEKEEPER_REPO_DEFAULT
    )
    GATEKEEPER_MODEL = resolve_hf_model_id(hf_gate)
    if _log_llm_catalog:
        print_hf_model_catalog(requested=(GATEKEEPER_MODEL,), role="Hugging Face (Gatekeeper)")
    print(f"Loading HF weights for Gatekeeper: {GATEKEEPER_MODEL} … (~10GB first time)")
    preload_gemma_hf(GATEKEEPER_MODEL, load_audio_variant=False)
else:
    _gate_requested = GATEKEEPER_MODEL
    GATEKEEPER_MODEL = resolve_model_id(
        _resolver_client,
        GATEKEEPER_MODEL,
        fallbacks=_GEMMA_GATEKEEPER_AI_FALLBACKS,
        role="Gatekeeper (AI Studio)",
    )
    if GATEKEEPER_MODEL != _gate_requested:
        print(
            f"  Gatekeeper model: {_gate_requested!r} not on this key — using {GATEKEEPER_MODEL!r}"
        )
    if any(x in GATEKEEPER_MODEL for x in ("26b", "31b", "27b")):
        print("  (heavier Gatekeeper model — yes/no triage may take 1–3 min per file)")

EMBEDDING_MODEL = resolve_model_id(
    _resolver_client,
    EMBEDDING_MODEL,
    fallbacks=("embeddinggemma-300m", "text-embedding-004", "gemini-embedding-001"),
    role="Embedding model",
)
SHIELD_MODEL = resolve_model_id(
    _resolver_client,
    SHIELD_MODEL,
    fallbacks=("shieldgemma-9b", "shieldgemma-2b"),
    role="Shield model (safety review)",
)

from colab_timed_steps import configure_pipeline_logging, heartbeat_interval
configure_pipeline_logging()
print(f"Pipeline logging: heartbeats every {heartbeat_interval():.0f}s (GOVERNANCE_HEARTBEAT_SECONDS)")

from gemma_hf_backend import demo4_use_huggingface, ensure_demo4_hf_ready, resolve_hf_model_id

if demo4_use_huggingface() and ACTIVE_MEDIA.run_demo4 and not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN required for Demo 4 (Gemma E2B audio/video on Hugging Face). "
        "Colab Secret HF_TOKEN + accept license at https://huggingface.co/google/gemma-4-E2B-it"
    )

if demo4_use_huggingface() and ACTIVE_MEDIA.run_demo4:
    DEMO4_MODEL = ensure_demo4_hf_ready()
    print(f"  Demo 4: Hugging Face {DEMO4_MODEL!r} (native audio/video — weights cached on Drive)")
elif model_requires_huggingface(GENAI_MODEL) or model_requires_huggingface(THINKING_MODEL):
    _d4_hf = os.environ.get("GOVERNANCE_DEMO4_MODEL", "").strip() or THINKING_MODEL
    DEMO4_MODEL = resolve_hf_model_id(_d4_hf)
else:
    from governance_meeting_llm import resolve_demo4_genai_model, list_demo4_capable_models_on_key
    _d4_requested = os.environ.get("GOVERNANCE_DEMO4_MODEL", "").strip()
    _d4_capable = list_demo4_capable_models_on_key(API_KEY, thinking_model=THINKING_MODEL)
    if _d4_capable:
        print("  Demo 4 audio-capable on this key:", ", ".join(_d4_capable[:8]))
    else:
        print("  Demo 4: no audio model on API key — set GOVERNANCE_DEMO4_USE_HF=1 (default)")
    DEMO4_MODEL = resolve_demo4_genai_model(
        GENAI_MODEL,
        gatekeeper_model=GATEKEEPER_MODEL,
        thinking_model=THINKING_MODEL,
        client=_resolver_client,
        api_key=API_KEY,
    )
    if _d4_requested and DEMO4_MODEL != _d4_requested:
        print(
            f"  Demo 4 model: {_d4_requested!r} not on this key — using {DEMO4_MODEL!r}"
        )

MAX_PDFS_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_PDFS_PER_JUR", "3"))
MAX_PAGES_PER_PDF = int(os.environ.get("GOVERNANCE_DEMO_MAX_PAGES_PER_PDF", "8"))
MAX_AUDIO_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_AUDIO_PER_JUR", "1"))
MAX_AUDIO_CHUNKS = int(os.environ.get("GOVERNANCE_DEMO_MAX_AUDIO_CHUNKS", "4"))
THINKING_BUDGET = int(os.environ.get("GOVERNANCE_DEMO_THINKING_BUDGET", "-1"))
DRIFT_FOCUS = os.environ.get("GOVERNANCE_DRIFT_FOCUS", "").strip() or None
_gate_max_env = os.environ.get("GOVERNANCE_GATEKEEPER_MAX_FILES", "").strip()
GATEKEEPER_MAX_FILES = int(_gate_max_env) if _gate_max_env else None


def _backend_label(model: str, *, gatekeeper: bool = False, demo4: bool = False) -> str:
    from gemma_hf_backend import gatekeeper_use_huggingface, demo4_use_huggingface, model_requires_huggingface
    if gatekeeper:
        return "Hugging Face" if gatekeeper_use_huggingface() else "Google AI Studio"
    if demo4 and demo4_use_huggingface():
        return "Hugging Face (E2B audio/video)"
    return "Hugging Face" if model_requires_huggingface(model) else "Google AI Studio"


print("LLM mode:", "all Hugging Face" if USE_HF_ALL else "hybrid (AI Studio default)")
if demo4_use_huggingface() and ACTIVE_MEDIA.run_demo4:
    print("Colab runtime: **L4 GPU** recommended for §6 (PDF + video / Demo 4 HF ~10GB VRAM).")
elif not gatekeeper_use_huggingface() and not model_requires_huggingface(GENAI_MODEL):
    print("Colab runtime: Gatekeeper + PDF use Google AI Studio; video still needs GPU + HF_TOKEN.")
print("GenAI model (demos 1/2 PDF):", GENAI_MODEL, f"({_backend_label(GENAI_MODEL)})")
print("Thinking model (demo 3):  ", THINKING_MODEL, f"({_backend_label(THINKING_MODEL)})")
print("Audio model (demo 4):     ", DEMO4_MODEL, f"({_backend_label(DEMO4_MODEL, demo4=True)})")
print(
    "Gatekeeper (triage):     ",
    GATEKEEPER_MODEL,
    f"({_backend_label(GATEKEEPER_MODEL, gatekeeper=True)})",
)
if gatekeeper_use_huggingface():
    print("  (GOVERNANCE_GATEKEEPER_FORCE_HF=1 — local E2B weights)")
print("Embedding (demo 6):      ", EMBEDDING_MODEL, "(Google AI Studio)")
print("Shield (safety pass):    ", SHIELD_MODEL, "(Google AI Studio)")
print(
    "Safety review:           ",
    "on (end of §6)" if os.environ.get("GOVERNANCE_SAFETY_REVIEW", "1").strip().lower()
    not in ("0", "false", "no", "off") else "off",
)
# Optional §3b — probe Google AI Studio (text stream hack + audio modality)
# os.environ["GOVERNANCE_DEMO4_USE_HF"] = "0"
# os.environ["GOVERNANCE_DEMO4_TRY_A4B_AUDIO"] = "1"
# os.environ["GOVERNANCE_DEMO4_MODEL"] = "gemma-4-26b-a4b-it"
# from governance_meeting_llm import print_probe_google_gemma_studio
# _probe_audio = next(iter(INVENTORY.audio), None) if "INVENTORY" in dir() else None
# print_probe_google_gemma_studio(GEMINI_API_KEY, audio_path=_probe_audio)

print(
    "Caps:",
    f"pdfs/jur={MAX_PDFS_PER_JUR}",
    f"pages/pdf={MAX_PAGES_PER_PDF}",
    f"audio/jur={MAX_AUDIO_PER_JUR}",
    f"chunks/audio={MAX_AUDIO_CHUNKS}",
    f"thinking_budget={THINKING_BUDGET}",
    f"gatekeeper_max_files={GATEKEEPER_MAX_FILES if GATEKEEPER_MAX_FILES is not None else 'unlimited'}",
)


§4: Colab detected — forcing notebook Colab Secrets first for API keys.
§4: no .env yet — trying Colab 🔑 Secrets for GEMINI_API_KEY / HF_TOKEN first
§4: Colab secret lookup failed for HF_TOKEN: TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.
   ⚠ Colab secret 'HF_TOKEN' unavailable (TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.). Falling back to env / .env.
   ⚠ Colab secret 'HF_TOKEN' unavailable (TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.). Use repo .env, os.environ, or Colab 🔑 Secrets.
§4: Colab secret lookup failed for GEMINI_API_KEY: TimeoutException: Requesting secret GEMINI_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.
§4: Colab secret lookup failed for GOOGLE_API_KEY: TimeoutException: Requesting secret GOOGLE_API_KEY timed out. Secrets c

RuntimeError: GEMINI_API_KEY missing. Colab: Colab 🔑 Secrets → GEMINI_API_KEY (notebook access ON), OR copy .env to /content/c1_gemma_4_good/.env, OR put .env in your hackathon folder on Google Drive.

## 5 · Inventory

Edit `SCOPE` in §2. **§6** runs gatekeeper + PDF + video on **one GPU runtime**.

On Colab, scoped jurisdictions are **mirrored to `/content/…`** when missing or stale. Set `GOVERNANCE_LOCAL_RAW_MIRROR=0` to stay on Drive only.


In [ ]:
from pathlib import Path
import os
import sys

_repo = PATHS.project_path
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from scripts.utils.gdrive_paths import resolve_governance_raw_inputs_root

DRIVE_RAW_ROOT = resolve_governance_raw_inputs_root(PIPE.root)
RAW_ROOT = DRIVE_RAW_ROOT
os.environ.setdefault("GOVERNANCE_LOCAL_RAW_MIRROR", "1")
os.environ.setdefault("GOVERNANCE_STEP_TIMING", "1")
os.environ.setdefault("GOVERNANCE_STEP_TIMESTAMPS", "1")
os.environ.setdefault("GOVERNANCE_PIPELINE_VERBOSE", "1")
os.environ.setdefault("GOVERNANCE_PIPELINE_PROGRESS", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_PROGRESS", "1")
PROCESSED_ROOT = PIPE.root / "03_processed_outputs"
GEMMA_JSON_ROOT = PROCESSED_ROOT / "02_gemma_json"
SUMMARIES_ROOT = PROCESSED_ROOT / "03_human_summaries"
SCRATCH_AUDIO_ROOT = PROCESSED_ROOT / "_scratch_audio_chunks"

try:
    from gemma_hf_backend import configure_hf_hub_cache, demo4_use_huggingface
    _hf_cache = os.environ.get("GOVERNANCE_HF_CACHE_DIR", "").strip()
    HF_CACHE_ROOT = Path(_hf_cache) if _hf_cache else (PROCESSED_ROOT / "_hf_hub_cache")
    configure_hf_hub_cache(HF_CACHE_ROOT)
    if demo4_use_huggingface():
        print(f"HF hub cache: {HF_CACHE_ROOT} (persists weights across Colab sessions)")
except ImportError:
    pass

for p in (GEMMA_JSON_ROOT, SUMMARIES_ROOT, SCRATCH_AUDIO_ROOT):
    p.mkdir(parents=True, exist_ok=True)

if not DRIVE_RAW_ROOT.is_dir():
    raise FileNotFoundError(
        f"Raw inputs root missing: {DRIVE_RAW_ROOT}\n"
        f"  Pipeline root: {PIPE.root}\n"
        f"  Scraped cache: {PATHS.project_path / 'data/cache/scraped_meetings'}\n"
        "Fix: copy to 01_raw_inputs on Drive, run 01_copy_scraped_meetings_cache_to_gdrive.py, "
        "set GOVERNANCE_RAW_INPUTS_ROOT, or populate data/cache/scraped_meetings locally."
    )

from demo_scope import get_active_preset, filter_inventories_for_scope, print_scope_plan

try:
    from meeting_date_scope import resolve_demo_meeting_dates_limit
    _demo_date_cap = resolve_demo_meeting_dates_limit()
except ImportError:
    _demo_date_cap = None

_DEMO_SCOPE = get_active_preset()
from colab_timed_steps import timed_step

with timed_step("Inventory walk on Drive (discover jurisdictions)"):
    _ALL_INVENTORIES = [inv for inv in walk_raw_inputs(DRIVE_RAW_ROOT) if inv.has_media]
    INVENTORIES = filter_inventories_for_scope(_ALL_INVENTORIES, _DEMO_SCOPE)

print(f"Drive raw inputs: {DRIVE_RAW_ROOT}")
print_scope_plan(_DEMO_SCOPE, _ALL_INVENTORIES, INVENTORIES)

if INVENTORIES:
    from colab_local_raw_mirror import mirror_inventories_to_local_raw

    INVENTORIES, RAW_ROOT = mirror_inventories_to_local_raw(
        INVENTORIES, DRIVE_RAW_ROOT
    )
print(f"Pipeline raw inputs (§6): {RAW_ROOT}")

if not INVENTORIES:
    print(
        "\nNo media found. Drop PDFs / audio under "
        f"{RAW_ROOT}/<STATE>/<county|municipality>/<jurisdiction_slug>/…"
    )


## 6 · Run pipeline (one GPU runtime)

1. **Runtime → L4 GPU** (or T4), **High RAM** if offered — use GPU for the **entire** notebook.
2. Colab **Secrets**: `GEMINI_API_KEY` and `HF_TOKEN` (notebook access **ON**).
3. Run **§0 → §1 → §2 → §3 → §4 → §5**, then the **§6** code cell below.

Do **not** switch to CPU or change runtime mid-run. If the session restarts, re-run **§0 → §1 → §5 → §6**.

Live log: `00_logs/gatekeeper_<jurisdiction>_*.log`


In [ ]:
# §6 — Run pipeline (PDF + video on GPU)
import importlib
import os
import sys
from pathlib import Path

os.environ.setdefault("GOVERNANCE_COLAB_SINGLE_RUNTIME", "1")

_repo = Path(os.environ.get("OPEN_NAVIGATOR_ROOT", "/content/c1_gemma_4_good"))
_utils = _repo / "scripts" / "colab" / "utils"
for _p in (_repo, _repo / "scripts" / "colab", _utils, _repo / "scripts" / "colab" / "runtime"):
    if _p.is_dir() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from colab_session_restore import restore_colab_session
from notebook_section6 import bootstrap_section6_runtime

restore_colab_session()
_repo, _crp, _nb6 = bootstrap_section6_runtime()
importlib.reload(_crp)

from colab_runtime_phases import apply_media_scope_for_phase
from colab_demos import DemoContext
from governance_meeting_llm import inventory_for_jurisdiction
from jurisdiction_pipeline import JurisdictionRunContext, run_governance_pipeline

apply_media_scope_for_phase("all")
os.environ.setdefault("GOVERNANCE_FORCE_REPROCESS", "0")
print("\n--- §6 Judge: full pipeline on GPU (PDF + video) ---\n")

if not INVENTORIES:
    raise RuntimeError("INVENTORIES empty — re-run §5 after §1 judge sync.")

_demo_ctx = DemoContext(
    api_key=API_KEY,
    genai_model=GENAI_MODEL,
    thinking_model=THINKING_MODEL,
    demo4_model=DEMO4_MODEL,
    gatekeeper_model=GATEKEEPER_MODEL,
    raw_root=RAW_ROOT,
    processed_root=PROCESSED_ROOT,
    gemma_json_root=GEMMA_JSON_ROOT,
    summaries_root=SUMMARIES_ROOT,
    scratch_audio_root=SCRATCH_AUDIO_ROOT,
    policy_prompt=POLICY_PROMPT,
    max_pdfs_per_jur=MAX_PDFS_PER_JUR,
    max_pages_per_pdf=MAX_PAGES_PER_PDF,
    max_audio_per_jur=MAX_AUDIO_PER_JUR,
    max_audio_chunks=MAX_AUDIO_CHUNKS,
    thinking_budget=THINKING_BUDGET,
    drift_focus=DRIFT_FOCUS,
)
_run_ctx = JurisdictionRunContext(
    raw_root=RAW_ROOT,
    pipe_root=PIPE.root,
    api_key=API_KEY,
    gatekeeper_model=GATEKEEPER_MODEL,
    demo_ctx=_demo_ctx,
    shield_model=SHIELD_MODEL,
    demo_date_cap=_demo_date_cap,
    gatekeeper_max_files=GATEKEEPER_MAX_FILES,
    run_safety_review=os.environ.get("GOVERNANCE_SAFETY_REVIEW", "1").strip().lower()
    not in ("0", "false", "no", "off"),
)
run_governance_pipeline(INVENTORIES, _run_ctx)
print("\n✓ Judge run complete. Outputs under", PROCESSED_ROOT)


## Optional — reruns only

Already covered in **§6** unless you need to regenerate outputs.

### 7 · Demo 1 — Scanned PDFs

Already run in **§6** (pipeline). Re-run only to regenerate OCR for capped PDFs.


In [ ]:
# Demo 1 — Native multimodality / visual document parsing.
# Probes the digital-text layer of every PDF and uses Gemma 4 to OCR scans.
from datetime import datetime, timezone

DEMO1_SYSTEM = (
    "You are a careful document transcription engine. Faithfully transcribe every "
    "word and number on each page of the attached PDF in reading order. Preserve "
    "table structure with vertical bars and dashes. Do not paraphrase, summarize, "
    "or invent content."
)
DEMO1_USER = (
    "Transcribe the attached PDF page by page. Begin each page with a heading line "
    "'### Page <n>' on its own line. If a page is blank, write '(blank page)'."
)

demo1_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    pdfs = inv.pdfs[:MAX_PDFS_PER_JUR]
    if not pdfs:
        continue
    print(f"\n— {j.relative_label} — {len(pdfs)} PDF(s)")
    for pdf in pdfs:
        try:
            digital_chars = len(extract_pdf_digital_text(pdf))
        except Exception as e:
            print(f"  ! {pdf.name}: PDF probe failed — {e}")
            continue
        scanned = digital_chars < 200
        tag = "SCANNED (dark data)" if scanned else f"digital ({digital_chars} chars)"
        print(f"  • {pdf.name}: {tag}")

        try:
            result = call_google_genai_multimodal(
                api_key=API_KEY,
                model=GENAI_MODEL,
                system_instruction=DEMO1_SYSTEM,
                user_text=DEMO1_USER,
                media=[(pdf, "application/pdf")],
                temperature=0.0,
                max_output_tokens=8192,
                media_resolution=TOKEN_BUDGET_HIGH if scanned else None,
            )
        except Exception as e:
            print(f"    ! Gemma call failed: {e}")
            continue

        out_txt = mirror_output_path(
            input_path=pdf,
            raw_root=RAW_ROOT,
            processed_root=GEMMA_JSON_ROOT,
            suffix=".visual_ocr.txt",
        )
        out_txt.write_text(result.text or "(empty response)", encoding="utf-8")
        demo1_report.append(
            {
                "jurisdiction": j.relative_label,
                "fips": j.fips,
                "pdf": str(pdf.relative_to(RAW_ROOT)),
                "scanned": scanned,
                "digital_chars": digital_chars,
                "output": str(out_txt.relative_to(PROCESSED_ROOT)),
                "model_chars": len(result.text or ""),
            }
        )
        print(f"    → {out_txt.relative_to(PROCESSED_ROOT)} ({len(result.text or '')} chars)")

scanned_count = sum(1 for r in demo1_report if r["scanned"])
print(
    f"\nDemo 1 done: {len(demo1_report)} PDFs, {scanned_count} flagged scanned "
    "→ Gemma 4 visual OCR recovered text that pdftotext could not."
)

import json as _json
demo1_summary_path = GEMMA_JSON_ROOT / "_demo1_visual_ocr_report.json"
demo1_summary_path.write_text(_json.dumps(demo1_report, indent=2), encoding="utf-8")
print("Report:", demo1_summary_path.relative_to(PROCESSED_ROOT))

### 8 · Demo 3 — Policy deconstruction

**Show judges:** `*.thinking.thoughts.md` next to the JSON — Gemma's planning on a real minutes PDF (e.g. Tuscaloosa nuisance demolitions: safety narrative vs. property-rights dissent).

Demo 3: **`THINKING_MODEL`** (`gemma-4-31b-it`). Demos 1/2 PDF: **`GENAI_MODEL`**. Demo 4: **`DEMO4_MODEL`** on **Hugging Face E2B/E4B** (`GOVERNANCE_DEMO4_USE_HF=1`, Opus chunks cached under `_scratch_audio_chunks`).


In [ ]:
# Demo 3 — Built-in thinking mode on the deconstruction prompt.
# One representative PDF per jurisdiction is processed with include_thoughts=True.
import json as _json

DEMO3_SYSTEM = (
    "You are an expert political scientist and data architect specializing in "
    "local governance. Follow the user's instructions exactly; preserve JSON validity."
)

_PRIORITY_PATTERNS = (
    "demolition", "demolitions", "nuisance",
    "minutes", "regular_session", "regular-session", "regular session",
    "council", "commission", "board", "hearing",
)


def _pick_representative_pdf(pdfs):
    if not pdfs:
        return None
    scored = []
    for p in pdfs:
        name = p.name.lower()
        score = 0
        for tag in _PRIORITY_PATTERNS:
            if tag in name:
                score += 5
        try:
            score += min(p.stat().st_size // 50_000, 50)
        except OSError:
            pass
        scored.append((score, p))
    scored.sort(key=lambda t: (-t[0], t[1].name))
    return scored[0][1]


demo3_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    pdf = _pick_representative_pdf(inv.pdfs)
    if pdf is None:
        continue
    print(f"\n— {j.relative_label}: {pdf.name}")

    geo_hint = (
        f"Geography hint from folder layout: state_code={j.state_code}, "
        f"scope={j.scope}, fips_or_place_id={j.fips or 'unknown'}. "
        "Use this when populating county_fips / county / postal_code in each decision."
    )
    user_text = (
        f"{POLICY_PROMPT}\n\n---\n{geo_hint}\n\n"
        "The attached PDF contains the meeting record. Apply the full deconstruction "
        "prompt to it. Stick to what is actually in the document."
    )

    try:
        result = call_google_genai_multimodal(
            api_key=API_KEY,
            model=GENAI_MODEL,
            system_instruction=DEMO3_SYSTEM,
            user_text=user_text,
            media=[(pdf, "application/pdf")],
            temperature=0.1,
            max_output_tokens=8192,
            media_resolution=TOKEN_BUDGET_HIGH,
            include_thoughts=True,
            thinking_budget=THINKING_BUDGET,
        )
    except Exception as e:
        print(f"  ! Gemma call failed: {e}")
        continue

    parsed = parse_policy_analysis_response(result.text or "")

    raw_out = mirror_output_path(
        input_path=pdf, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
        suffix=".thinking.raw.txt",
    )
    raw_out.write_text(result.text or "", encoding="utf-8")

    if parsed.get("json_analysis") is not None:
        json_out = mirror_output_path(
            input_path=pdf, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
            suffix=".thinking.json",
        )
        json_out.write_text(
            _json.dumps(parsed["json_analysis"], indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        print(f"  → {json_out.relative_to(PROCESSED_ROOT)}")

    if parsed.get("summary"):
        summary_out = mirror_output_path(
            input_path=pdf, raw_root=RAW_ROOT, processed_root=SUMMARIES_ROOT,
            suffix=".thinking.summary.md",
        )
        summary_out.write_text(parsed["summary"], encoding="utf-8")
        print(f"  → {summary_out.relative_to(PROCESSED_ROOT)}")

    if result.thoughts:
        thoughts_out = mirror_output_path(
            input_path=pdf, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
            suffix=".thinking.thoughts.md",
        )
        thoughts_out.write_text(result.thoughts, encoding="utf-8")
        print(
            f"  → {thoughts_out.relative_to(PROCESSED_ROOT)} "
            f"(reasoning trace: {len(result.thoughts)} chars)"
        )
    else:
        print("  (no thoughts returned — model may not have surfaced a trace this run)")

    demo3_report.append(
        {
            "jurisdiction": j.relative_label,
            "pdf": str(pdf.relative_to(RAW_ROOT)),
            "thoughts_chars": len(result.thoughts or ""),
            "json_ok": parsed.get("json_analysis") is not None and "_error" not in (parsed.get("json_analysis") or {}),
            "parse_error": parsed.get("parse_error"),
        }
    )

if demo3_report:
    ok = sum(1 for r in demo3_report if r["json_ok"])
    print(
        f"\nDemo 3 done: {len(demo3_report)} PDFs deconstructed, "
        f"{ok} produced parseable JSON. Reasoning trace saved alongside each output."
    )
else:
    print("\nDemo 3: no PDFs available — check the walker output above.")

### 9 · Demo 4 — Audio drift

Prefer **§6** (`run_governance_pipeline`) — it uses Hugging Face E2B + Opus chunks via `colab_demos.run_demo4`. This cell is a thin rerun only when §6 skipped Demo 4.

**Show judges:** `policy_drift.json` and `policy_drift.mmd` under each audio folder — how one issue moves across 15‑minute chunks of a long meeting.

Uses agenda+minutes text (when organized in §4) to ground names in the audio prompt. Requires `GOVERNANCE_DEMO4_USE_HF=1` (default), GPU, and `HF_TOKEN`. Optional focus: `GOVERNANCE_DRIFT_FOCUS`.


In [ ]:
# Demo 4 — Long-meeting chunking + Policy Drift Detector.
# Splits audio into 15-minute chunks, runs policy_analysis_v1 per chunk, then runs the drift pass.
import json as _json

DEMO4_SYSTEM = (
    "You are an expert political scientist analyzing one chunk of a long meeting. "
    "Follow the user's instructions exactly; preserve JSON validity. The chunk_index "
    "tells you which 15-minute slice of the meeting this audio covers."
)

try:
    from meeting_grouping import (
        build_meeting_collateral_brief,
        format_audio_analysis_prompt,
        resolve_meeting_dir,
    )
except ImportError:
    build_meeting_collateral_brief = None

_brief_cache = {}

demo4_report = []
_demo4_force = force_reprocess_outputs()
for inv in INVENTORIES:
    j = inv.jurisdiction
    audios = select_demo4_media(inv.audio, RAW_ROOT, max_files=MAX_AUDIO_PER_JUR)
    if not audios:
        continue
    print(f"\n— {j.relative_label}: {len(audios)} audio file(s)")
    for audio in audios:
        print(f"  • {audio.name}")
        per_audio_dir = mirror_output_path(
            input_path=audio, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT, suffix="",
        )
        per_audio_dir.mkdir(parents=True, exist_ok=True)

        drift_out = per_audio_dir / "policy_drift.json"
        from governance_meeting_llm import (
            demo4_media_work_dir,
            demo4_prefer_opus_chunks,
            demo4_use_video_chunks,
            discover_demo4_chunk_media,
            find_existing_demo4_chunks,
            iter_demo4_ingest_strategies,
            model_supports_video_input,
        )

        scratch = demo4_media_work_dir(SCRATCH_AUDIO_ROOT, RAW_ROOT, audio)
        use_video = (
            audio.suffix.lower() in {".mp4", ".mov", ".webm", ".mkv"}
            and demo4_use_video_chunks()
            and model_supports_video_input(DEMO4_MODEL)
            and not demo4_prefer_opus_chunks()
        )
        existing_chunks = find_existing_demo4_chunks(scratch, video=use_video)
        if existing_chunks and not _demo4_force:
            chunk_media = discover_demo4_chunk_media(scratch, video=use_video)[:MAX_AUDIO_CHUNKS]
            print(f"    reuse {len(existing_chunks)} ffmpeg chunk(s) from scratch")
        else:
            ingest_plans = list(
                iter_demo4_ingest_strategies(
                    audio,
                    out_dir=scratch,
                    chunk_minutes=15,
                    prefer_video=use_video,
                    demo4_model=DEMO4_MODEL,
                )
            )
            if not ingest_plans:
                print("    ! ffmpeg could not produce audio slices (Opus or MP3)")
                continue
            _lbl, chunk_media = ingest_plans[0]
            chunk_media = chunk_media[:MAX_AUDIO_CHUNKS]
            print(f"    ingest {_lbl}: {len(chunk_media)} chunk(s) (cap MAX_AUDIO_CHUNKS={MAX_AUDIO_CHUNKS})")
        if not chunk_media:
            continue
        print(f"    {len(chunk_media)} chunk(s) of 15 min each (cap MAX_AUDIO_CHUNKS={MAX_AUDIO_CHUNKS})")

        chunk_jsons = []
        _chunks_regenerated = False
        for idx, (chunk_path, chunk_mime) in enumerate(chunk_media):
            chunk_out = per_audio_dir / f"chunk_{idx:03d}.json"
            if not _demo4_force and policy_chunk_output_complete(chunk_out):
                data = read_json_file(chunk_out) or {}
                chunk_jsons.append(data.get("json_analysis") or {})
                print(f"    chunk {idx}: reuse existing JSON")
                continue
            _chunks_regenerated = True
            geo_hint = (
                f"Geography hint: state_code={j.state_code}, scope={j.scope}, "
                f"fips_or_place_id={j.fips or 'unknown'}. chunk_index={idx} of {len(chunk_media)}. "
            )
            meeting_dir = (
                resolve_meeting_dir(audio, RAW_ROOT) if build_meeting_collateral_brief else None
            )
            brief = ""
            if meeting_dir and build_meeting_collateral_brief:
                mk = str(meeting_dir)
                if mk not in _brief_cache:
                    _brief_cache[mk] = build_meeting_collateral_brief(
                        meeting_dir,
                        api_key=API_KEY,
                        model=GENAI_MODEL,
                        client=None,
                    )
                brief = _brief_cache.get(mk) or ""
            chunk_hint = (
                "The attached audio is one 15-minute slice of a longer governance meeting. "
                "Apply the deconstruction prompt to what is audible. Use the chunk_index "
                "to anchor the timeline and assign consistent subject_id slugs across chunks."
            )
            user_text = format_audio_analysis_prompt(
                policy_prompt=POLICY_PROMPT,
                meeting_brief=brief,
                geo_hint=geo_hint,
                chunk_hint=chunk_hint,
            ) if brief and build_meeting_collateral_brief else (
                f"{POLICY_PROMPT}\n\n---\n{geo_hint}\n\n{chunk_hint}"
            )
            try:
                result = call_google_genai_multimodal(
                    api_key=API_KEY,
                    model=DEMO4_MODEL,
                    system_instruction=DEMO4_SYSTEM,
                    user_text=user_text,
                    media=[(chunk_path, chunk_mime)],
                    temperature=0.1,
                    max_output_tokens=8192,
                    demo4=True,
                )
            except Exception as e:
                print(f"    ! chunk {idx} failed: {e}")
                continue
            parsed = parse_policy_analysis_response(result.text or "")
            chunk_out.write_text(
                _json.dumps(
                    {
                        "chunk_index": idx,
                        "audio_source": str(chunk_path.name),
                        "json_analysis": parsed.get("json_analysis"),
                        "summary": parsed.get("summary"),
                        "parse_error": parsed.get("parse_error"),
                    },
                    indent=2,
                    ensure_ascii=False,
                ),
                encoding="utf-8",
            )
            chunk_jsons.append(parsed.get("json_analysis") or {})
            print(f"    chunk {idx}: → {chunk_out.relative_to(PROCESSED_ROOT)}")

        if not chunk_jsons or not any(chunk_jsons):
            continue

        # Drift detector: Gemma 4 takes every chunk's JSON and reports per-subject
        # narrative drift along the five `narrative_analysis` dimensions defined in
        # `prompts/policy_analysis_v1.md` (dominant_narrative, dissenting_interpretations,
        # causal_interpretations, value_conflicts, tradeoff_analysis). The canonical
        # prompt body is pinned into context so the model honors the latest schema verbatim.
        if (
            not _demo4_force
            and not _chunks_regenerated
            and demo4_drift_output_complete(drift_out)
        ):
            drift = read_json_file(drift_out) or {}
            print(f"    drift detector: reuse existing → {drift_out.relative_to(PROCESSED_ROOT)}")
        else:
            drift = policy_drift_summarize(
                chunk_jsons,
                api_key=API_KEY,
                model=DEMO4_MODEL,
                thinking_model=THINKING_MODEL,
                focus_hint=DRIFT_FOCUS,
                canonical_prompt_text=POLICY_PROMPT,
            )
            drift_out.write_text(_json.dumps(drift, indent=2, ensure_ascii=False), encoding="utf-8")

        drifted = drift.get("subjects") or drift.get("drifted_subjects") or []

        # Concatenate per-subject Mermaid timelines so the .mmd file is a single
        # diagram per audio (legacy schema returned one top-level diagram_timeline).
        mmd_blocks = []
        for s in drifted:
            tl = s.get("diagram_timeline")
            if isinstance(tl, str) and tl.strip():
                label = s.get("subject_label") or s.get("subject_id") or "subject"
                mmd_blocks.append(f"%% {label}\n{tl}")
        legacy_tl = drift.get("diagram_timeline")
        if not mmd_blocks and isinstance(legacy_tl, str) and legacy_tl.strip():
            mmd_blocks.append(legacy_tl)
        if mmd_blocks:
            (per_audio_dir / "policy_drift.mmd").write_text(
                "\n\n".join(mmd_blocks), encoding="utf-8"
            )

        meeting_summary = drift.get("meeting_level_summary") or {}
        print(
            f"    drift detector: {len(drifted)} subject(s) tracked across {len(chunk_jsons)} chunks "
            f"→ {drift_out.relative_to(PROCESSED_ROOT)}"
        )
        if meeting_summary.get("headline"):
            print(f"      meeting headline: {meeting_summary['headline'][:140]}")
        for s in drifted[:5]:
            label = s.get("subject_label") or s.get("subject_id") or "?"
            headline = s.get("drift_headline") or s.get("drift_summary") or ""
            stability = (s.get("narrative_stability_assessment") or {}).get("narrative_novelty")
            stability_tag = f" [{stability}]" if stability else ""
            print(f"      · {label}{stability_tag}: {headline[:120]}")
        demo4_report.append(
            {
                "jurisdiction": j.relative_label,
                "audio": str(audio.relative_to(RAW_ROOT)),
                "chunks": len(chunk_jsons),
                "subjects_tracked": len(drifted),
                "subjects_with_drift": sum(1 for s in drifted if s.get("drift_observed")),
                "emergent_value_tensions": meeting_summary.get("emergent_value_tensions") or [],
            }
        )

if demo4_report:
    print(
        f"\nDemo 4 done: {len(demo4_report)} audio file(s) processed. "
        "Gemma's alternating local + global attention kept subject ids consistent "
        "across 15-minute chunks; drift detector reported per-subject drift along the "
        "five narrative_analysis dimensions from policy_analysis_v1.md."
    )
else:
    print("\nDemo 4: no audio files in the inventory. Drop .mp3/.opus/.wav or video (.mp4/.webm/…) under a jurisdiction folder; video is transcoded to Opus before analysis.")


### 9a · Plain transcript (skip for fast run)

**§6 does not run this.** Word-for-word text lands in `02_gemma_json/…/transcript.en.txt` (not `01_transcripts/`). Reuses Demo 4 ffmpeg slices (Opus/MP3). Needs `GOVERNANCE_DEMO4_USE_HF=1` + GPU for audio (`DEMO4_MODEL`). Default language: `en` (`GOVERNANCE_TRANSCRIPTION_LANGUAGES`).


In [ ]:
# Demo 4a — Plain Transcription per audio file.
# Mirrors Demo 4's audio iteration so transcripts land beside the chunk JSONs.
_TRANSCRIPTION_LANGS_RAW = os.environ.get("GOVERNANCE_TRANSCRIPTION_LANGUAGES", "en")
TRANSCRIPTION_LANGS = [
    lang.strip()
    for lang in _TRANSCRIPTION_LANGS_RAW.split(",")
    if lang.strip()
]
# Validate up-front so a typo in the env var fails fast, not after N audio calls.
_SUPPORTED_LANG_TAGS = {tag.lower() for tag in TRANSCRIPTION_SUPPORTED_LANGUAGES.values()}
_invalid_langs = [
    lang for lang in TRANSCRIPTION_LANGS if lang.lower() not in _SUPPORTED_LANG_TAGS
]
if _invalid_langs:
    raise RuntimeError(
        f"GOVERNANCE_TRANSCRIPTION_LANGUAGES contains unsupported tag(s) {_invalid_langs}. "
        f"Supported: {sorted(_SUPPORTED_LANG_TAGS)} (currently {list(TRANSCRIPTION_SUPPORTED_LANGUAGES)})."
    )

_TRANSCRIPTION_MAX_CHUNKS = int(
    os.environ.get("GOVERNANCE_TRANSCRIPTION_MAX_CHUNKS", str(MAX_AUDIO_CHUNKS))
)

demo4a_report = []
_demo4a_force = force_reprocess_outputs()
if not TRANSCRIPTION_LANGS:
    print("Demo 4a: GOVERNANCE_TRANSCRIPTION_LANGUAGES is empty — skipping.")
else:
    print(f"Demo 4a: transcribing in {TRANSCRIPTION_LANGS} (cap {_TRANSCRIPTION_MAX_CHUNKS} chunks/file)")
    for inv in INVENTORIES:
        j = inv.jurisdiction
        audios = inv.audio[:MAX_AUDIO_PER_JUR]
        if not audios:
            continue
        print(f"\n— {j.relative_label}: {len(audios)} audio file(s)")
        for audio in audios:
            print(f"  • {audio.name}")
            per_audio_dir = mirror_output_path(
                input_path=audio, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT, suffix="",
            )
            per_audio_dir.mkdir(parents=True, exist_ok=True)

            from governance_meeting_llm import (
                demo4_media_work_dir,
                demo4_prefer_opus_chunks,
                demo4_use_video_chunks,
                discover_demo4_chunk_media,
                find_existing_demo4_chunks,
                iter_demo4_ingest_strategies,
                model_supports_video_input,
            )

            scratch = demo4_media_work_dir(SCRATCH_AUDIO_ROOT, RAW_ROOT, audio)
            use_video = (
                audio.suffix.lower() in {".mp4", ".mov", ".webm", ".mkv"}
                and demo4_use_video_chunks()
                and model_supports_video_input(DEMO4_MODEL)
                and not demo4_prefer_opus_chunks()
            )
            existing = find_existing_demo4_chunks(scratch, video=use_video)
            if existing:
                chunk_media = discover_demo4_chunk_media(scratch, video=use_video)[
                    :_TRANSCRIPTION_MAX_CHUNKS
                ]
                print(f"    reusing {len(existing)} ffmpeg chunk(s) from scratch")
            else:
                ingest_plans = list(
                    iter_demo4_ingest_strategies(
                        audio,
                        out_dir=scratch,
                        chunk_minutes=15,
                        prefer_video=use_video,
                        demo4_model=DEMO4_MODEL,
                    )
                )
                if not ingest_plans:
                    print("    ! ffmpeg could not produce audio slices")
                    continue
                _lbl, chunk_media = ingest_plans[0]
                chunk_media = chunk_media[:_TRANSCRIPTION_MAX_CHUNKS]
            if not chunk_media:
                continue
            print(f"    {len(chunk_media)} chunk(s) to transcribe")

            for lang in TRANSCRIPTION_LANGS:
                out_path = per_audio_dir / f"transcript.{lang}.txt"
                if not _demo4a_force and text_output_complete(out_path):
                    print(f"    [{lang}] reuse existing → {out_path.relative_to(PROCESSED_ROOT)}")
                    demo4a_report.append(
                        {
                            "jurisdiction": j.relative_label,
                            "audio": str(audio.relative_to(RAW_ROOT)),
                            "language": lang,
                            "chunks": len(chunk_media),
                            "chars": len(out_path.read_text(encoding="utf-8")),
                            "errors": 0,
                            "reused": True,
                        }
                    )
                    continue
                pieces = []
                errors = 0
                for idx, (chunk_path, chunk_mime) in enumerate(chunk_media):
                    try:
                        text = transcribe_audio_with_gemma(
                            api_key=API_KEY,
                            model=DEMO4_MODEL,
                            audio_path=chunk_path,
                            language=lang,
                            mime_type=chunk_mime,
                        )
                    except Exception as e:
                        errors += 1
                        print(f"    ! chunk {idx} [{lang}] failed: {e}")
                        continue
                    if text:
                        pieces.append(text)

                if not pieces:
                    print(f"    [{lang}] no text produced — skipping write")
                    continue

                # Join chunk transcripts with single spaces. The model is instructed
                # not to emit newlines; we still collapse defensively in the helper.
                transcript = " ".join(pieces)
                out_path = per_audio_dir / f"transcript.{lang}.txt"
                out_path.write_text(transcript, encoding="utf-8")
                print(
                    f"    [{lang}] → {out_path.relative_to(PROCESSED_ROOT)} "
                    f"({len(transcript):,} chars from {len(pieces)}/{len(chunk_media)} chunks"
                    + (f", {errors} error(s)" if errors else "")
                    + ")"
                )
                demo4a_report.append(
                    {
                        "jurisdiction": j.relative_label,
                        "audio": str(audio.relative_to(RAW_ROOT)),
                        "language": lang,
                        "chunks_transcribed": len(pieces),
                        "chunks_attempted": len(chunk_media),
                        "errors": errors,
                        "chars": len(transcript),
                    }
                )

if demo4a_report:
    print(
        f"\nDemo 4a done: {len(demo4a_report)} transcript file(s) written. "
        "Plain text is now beside each audio's chunk JSONs for citation / human review."
    )
else:
    print("\nDemo 4a: no transcripts written (no audio in inventory or all chunks failed).")

### 10 · Contact photos

Classifies `_contact_images/` as person vs. logo/building; writes JSON beside each image.


In [ ]:
# Demo 5 — Contact image enrichment. For each image found under a jurisdiction,
# decide whether it is a person; return perceived demographics or a subject tag.
import json as _json

DEMO5_SYSTEM = (
    "You are a careful image triage assistant for a local-government open-data "
    "pipeline. You analyze public contact photos of elected officials and "
    "department heads scraped from public government websites. You always return "
    "strict JSON only. You never claim certainty about demographic attributes — "
    "every attribute is explicitly perceived from the image and you return null "
    "when the image is too low-resolution, partially occluded, ambiguous, or when "
    "you would be guessing rather than observing."
)
DEMO5_USER = (
    "Look at the attached image and decide whether it depicts a single identifiable "
    "person. Return JSON with this shape:\n\n"
    "{\n"
    "  \"is_person\": bool,\n"
    "  \"confidence\": float between 0.0 and 1.0,\n"
    "  \"perceived\": {\n"
    "    \"age_range\": \"one of: child, teen, 20-29, 30-39, 40-49, 50-59, 60-69, 70-plus, or null\",\n"
    "    \"race\": \"perceived racial category or null\",\n"
    "    \"gender\": \"perceived gender presentation or null\",\n"
    "    \"ethnicity\": \"perceived ethnic background or null\",\n"
    "    \"demeanor\": \"one of: formal, smiling, neutral, stern, candid, or null\"\n"
    "  } or null when is_person is false,\n"
    "  \"subject_tag\": \"short descriptor when is_person is false (e.g. 'city logo', "
    "'aerial photo of courthouse', 'organizational chart'); otherwise null\",\n"
    "  \"notes\": \"short caveat — e.g. 'low resolution', 'side profile', "
    "'multiple people visible', 'official portrait crop'\"\n"
    "}\n\n"
    "Rules:\n"
    "- These are PUBLIC officials in a PUBLIC-records transparency context.\n"
    "- Every 'perceived.*' field is a best visual estimate, not a factual claim.\n"
    "- Return null for any 'perceived' subfield where you would be guessing.\n"
    "- If multiple people appear, set is_person=false and use subject_tag='group photo' (and notes).\n"
    "- Return only the JSON object, no prose or markdown."
)


def _mime_for_image(p):
    ext = p.suffix.lower()
    if ext in (".jpg", ".jpeg"): return "image/jpeg"
    if ext == ".png": return "image/png"
    if ext == ".webp": return "image/webp"
    if ext == ".gif": return "image/gif"
    if ext == ".bmp": return "image/bmp"
    return "application/octet-stream"


MAX_IMAGES_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_IMAGES_PER_JUR", "12"))

demo5_report = []
if MAX_IMAGES_PER_JUR <= 0:
    print("\nDemo 5 skipped (Fast scope — pick Standard or Full in §2 for contact images).")
for inv in ([] if MAX_IMAGES_PER_JUR <= 0 else INVENTORIES):
    j = inv.jurisdiction
    if not inv.images:
        continue
    images = inv.images[:MAX_IMAGES_PER_JUR]
    print(f"\n— {j.relative_label}: {len(images)} image(s) (cap={MAX_IMAGES_PER_JUR})")
    for img in images:
        try:
            result = call_google_genai_multimodal(
                api_key=API_KEY,
                model=GENAI_MODEL,
                system_instruction=DEMO5_SYSTEM,
                user_text=DEMO5_USER,
                media=[(img, _mime_for_image(img))],
                temperature=0.0,
                max_output_tokens=1024,
                media_resolution=TOKEN_BUDGET_HIGH,
            )
        except Exception as e:
            print(f"  ! {img.name}: Gemma call failed — {e}")
            continue

        try:
            payload = _json.loads((result.text or "").strip().lstrip("`"))
        except Exception:
            payload = {"_parse_error": True, "_raw": (result.text or "")[:2000]}

        out_path = mirror_output_path(
            input_path=img, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
            suffix=".image_triage.json",
        )
        out_path.write_text(
            _json.dumps(
                {
                    "image": str(img.relative_to(RAW_ROOT)),
                    "jurisdiction_id": j.jurisdiction_id,
                    "model": GENAI_MODEL,
                    "result": payload,
                },
                indent=2, ensure_ascii=False,
            ),
            encoding="utf-8",
        )
        is_person = bool(payload.get("is_person")) if isinstance(payload, dict) else False
        if is_person:
            perceived = payload.get("perceived") or {}
            details = ", ".join(
                f"{k}={v}" for k, v in perceived.items() if v not in (None, "")
            ) or "(no observable attributes)"
            print(f"  · {img.name}: PERSON — {details}")
        else:
            tag = payload.get("subject_tag") if isinstance(payload, dict) else None
            print(f"  · {img.name}: non-person → {tag or '(no tag)'}")
        demo5_report.append(
            {
                "jurisdiction": j.relative_label,
                "jurisdiction_id": j.jurisdiction_id,
                "image": str(img.relative_to(RAW_ROOT)),
                "is_person": is_person,
                "output": str(out_path.relative_to(PROCESSED_ROOT)),
            }
        )

if demo5_report:
    persons = sum(1 for r in demo5_report if r["is_person"])
    print(
        f"\nDemo 5 done: {len(demo5_report)} images classified across "
        f"{len({r['jurisdiction'] for r in demo5_report})} jurisdictions. "
        f"{persons} persons / {len(demo5_report) - persons} non-person subjects."
    )
else:
    print("\nDemo 5: no images in the inventory (check _contact_images/ folders).")

### 11 · Merge reference JSON

Joins Gemma outputs with meeting-data / contacts reference files by `jurisdiction_id`.


In [ ]:
# Walk every *.json under GEMMA_JSON_ROOT and attach the matching meeting-data
# and contacts rows by jurisdiction_id, derived from the mirrored output path.
import json as _json


def _jurisdiction_id_from_output(path):
    """Mirror layout is <STATE>/<scope>/<jur_slug>/...  →  jurisdiction_<state>_<scope>_<tail>."""
    try:
        rel = path.resolve().relative_to(GEMMA_JSON_ROOT.resolve())
    except ValueError:
        return None
    parts = rel.parts
    if len(parts) < 3:
        return None
    state, scope, slug = parts[0].lower(), parts[1].lower(), parts[2].lower()
    scope_prefix = f"{scope}_"
    tail = slug[len(scope_prefix):] if slug.startswith(scope_prefix) else slug
    return f"jurisdiction_{state}_{scope}_{tail}"


meeting_data = load_meeting_data_lookup(PIPE.meeting_data_by_jurisdiction_id)
contacts = load_contacts_lookup(PIPE.contacts_by_jurisdiction_id)

if not meeting_data and not contacts:
    print(
        f"No reference lookups found under {PIPE.meeting_data_by_jurisdiction_id} "
        f"or {PIPE.contacts_by_jurisdiction_id} — skip."
    )
else:
    print(
        f"Loaded {len(meeting_data)} meeting_data and {len(contacts)} contacts "
        "rows keyed by jurisdiction_id."
    )
    merged_meeting = 0
    merged_contacts = 0
    missing_both = 0
    skipped = 0
    for jp in sorted(GEMMA_JSON_ROOT.rglob("*.json")):
        if jp.name.startswith("_") or jp.name.startswith("policy_drift"):
            continue
        try:
            data = _json.loads(jp.read_text(encoding="utf-8"))
        except _json.JSONDecodeError:
            skipped += 1
            continue
        jid = _jurisdiction_id_from_output(jp)
        if not jid:
            continue
        analysis = data
        if isinstance(data, dict) and isinstance(data.get("json_analysis"), dict):
            analysis = data["json_analysis"]
        if not isinstance(analysis, dict):
            continue
        had_meeting = jid in meeting_data
        had_contacts = jid in contacts
        if not (had_meeting or had_contacts):
            missing_both += 1
            continue
        if had_meeting:
            merge_meeting_data_by_jurisdiction(analysis, meeting_data, jid)
            merged_meeting += 1
        if had_contacts:
            merge_contacts_by_jurisdiction(analysis, contacts, jid)
            merged_contacts += 1
        jp.write_text(_json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
        tags = []
        if had_meeting:
            tags.append("meeting_data")
        if had_contacts:
            tags.append("contacts")
        print(f"  enriched: {jp.relative_to(PROCESSED_ROOT)}  jurisdiction_id={jid}  +{','.join(tags)}")
    print(
        f"\nMerge done: meeting_data={merged_meeting}, contacts={merged_contacts}, "
        f"no-match jurisdictions={missing_both}, unparseable files={skipped}."
    )

### 12 · Cross-jurisdiction clustering

EmbeddingGemma finds similar policy language across Tuscaloosa and Big Timber. Needs `GEMINI_API_KEY` and prior demo JSON. Output: `04_embeddings/cross_jurisdiction_pairs.json`.


In [ ]:
# Demo 6 — EmbeddingGemma: cluster similar policy items across jurisdictions.
import json as _json

EMBED_MAX_ITEMS = int(os.environ.get("GOVERNANCE_EMBED_MAX_ITEMS", "200"))
EMBED_SIM_THRESHOLD = float(os.environ.get("GOVERNANCE_EMBED_SIM_THRESHOLD", "0.7"))
EMBED_REPORT_TOP_N = int(os.environ.get("GOVERNANCE_EMBED_REPORT_TOP_N", "30"))


def _harvest_snippets(json_path):
    """Pull high-signal text snippets from one Gemma JSON output."""
    try:
        data = _json.loads(json_path.read_text(encoding="utf-8"))
    except Exception:
        return []
    snippets = []
    payload = data
    if isinstance(data, dict) and isinstance(data.get("json_analysis"), dict):
        payload = data["json_analysis"]
    if not isinstance(payload, dict):
        return []
    # Demo 1 + 2 outputs carry raw_text / headlines.
    if isinstance(payload.get("raw_text"), str) and len(payload["raw_text"]) > 40:
        snippets.append(payload["raw_text"][:600])
    if isinstance(payload.get("headline"), str):
        snippets.append(payload["headline"])
    # Demo 3 outputs: subjects[], decisions[].
    for sub in payload.get("subjects", []) or []:
        name = sub.get("name") or sub.get("descriptive_name") or sub.get("subject_id")
        if name:
            snippets.append(str(name))
    for dec in payload.get("decisions", []) or []:
        title = dec.get("decision_title") or dec.get("title")
        if title:
            snippets.append(str(title))
    return [s.strip() for s in snippets if s and s.strip()]


items = []
for jp in sorted(GEMMA_JSON_ROOT.rglob("*.json")):
    if jp.name.startswith("_"):
        continue
    try:
        rel = jp.resolve().relative_to(GEMMA_JSON_ROOT.resolve())
    except ValueError:
        continue
    parts = rel.parts
    if len(parts) < 3:
        continue
    state, scope, slug = parts[0], parts[1], parts[2]
    jurisdiction = f"{state}/{scope}/{slug}"
    for text in _harvest_snippets(jp):
        items.append({"jurisdiction": jurisdiction, "source": str(rel), "text": text})
    if len(items) >= EMBED_MAX_ITEMS:
        break

items = items[:EMBED_MAX_ITEMS]
print(f"Embedding {len(items)} snippet(s) from {GEMMA_JSON_ROOT}")

if not items:
    print("Demo 6: no JSON snippets to embed yet — run Demos 1–3 first.")
else:
    if not GEMINI_API_KEY:
        raise RuntimeError("Demo 6 needs GEMINI_API_KEY (AI Studio) for EmbeddingGemma, or skip this cell.")
    vectors = embed_text_with_gemma(
        api_key=GEMINI_API_KEY,
        model=EMBEDDING_MODEL,
        texts=[it["text"] for it in items],
    )
    for it, v in zip(items, vectors):
        it["embedding_dims"] = len(v)

    embeddings_root = PIPE.root / "03_processed_outputs" / "04_embeddings"
    embeddings_root.mkdir(parents=True, exist_ok=True)
    items_path = embeddings_root / "items.jsonl"
    items_path.write_text(
        "\n".join(_json.dumps(it, ensure_ascii=False) for it in items) + "\n",
        encoding="utf-8",
    )

    sim = cosine_similarity_matrix(vectors)
    pairs = []
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i]["jurisdiction"] == items[j]["jurisdiction"]:
                continue  # we want CROSS-jurisdiction matches
            if sim[i][j] < EMBED_SIM_THRESHOLD:
                continue
            pairs.append({
                "similarity": round(float(sim[i][j]), 4),
                "left": {"jurisdiction": items[i]["jurisdiction"], "text": items[i]["text"][:200]},
                "right": {"jurisdiction": items[j]["jurisdiction"], "text": items[j]["text"][:200]},
            })
    pairs.sort(key=lambda p: -p["similarity"])
    pairs = pairs[:EMBED_REPORT_TOP_N]

    pairs_path = embeddings_root / "cross_jurisdiction_pairs.json"
    pairs_path.write_text(
        _json.dumps({"threshold": EMBED_SIM_THRESHOLD, "pairs": pairs}, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print(f"Demo 6 done — {len(pairs)} cross-jurisdiction pair(s) ≥ {EMBED_SIM_THRESHOLD}.")
    for p in pairs[:5]:
        print(f"  {p['similarity']:.3f}  {p['left']['jurisdiction']}  ↔  {p['right']['jurisdiction']}")
        print(f"            L: {p['left']['text'][:90]}")
        print(f"            R: {p['right']['text'][:90]}")
    print(f"  items   → {items_path.relative_to(PIPE.root)}")
    print(f"  pairs   → {pairs_path.relative_to(PIPE.root)}")


### 13 · Safety review (also runs at end of §6)

ShieldGemma reviews Demo 1–5 LLM outputs → `05_safety_review/`. **On by default** (`GOVERNANCE_SAFETY_REVIEW=1`). Set `0` to skip.


In [ ]:
# Rerun safety review without re-running §6 (same logic as end of pipeline).
from colab_safety_review import run_safety_review, safety_review_enabled

if safety_review_enabled():
    run_safety_review(
        api_key=API_KEY,
        shield_model=SHIELD_MODEL,
        gemma_json_root=GEMMA_JSON_ROOT,
        safety_root=PIPE.root / "03_processed_outputs" / "05_safety_review",
        summaries_root=SUMMARIES_ROOT,
    )
else:
    print("Safety review disabled (GOVERNANCE_SAFETY_REVIEW=0).")


## Reference

### Troubleshooting

| Issue | Fix |
|-------|-----|
| Runtime disconnected / crashed | Re-run **§0 → §1 → §5 → §6** on **L4 GPU** (same runtime — do not switch CPU/GPU mid-run). |
| “Connected to GPU but not utilizing it” | GPU spikes during Demo 4 / HF load; Gatekeeper + PDF mostly use Google AI API. |
| `ModuleNotFoundError` after restart | Re-run **§0 → §1** (repo path + corpus), then **§5 → §6**. |
| `GEMINI_API_KEY` missing | Colab Secrets or paste in §0; re-run **§4** then **§6**. |
| Logs on Colab but not on PC Drive | Normal sync lag. Tail **`/content/governance_pipeline_local/00_logs/gatekeeper_*.log`** (mirror, instant) or refresh Drive. |
| Demo 3 stuck on **Agenda** (7+ min, `AFC is enabled…`) | Default **`GOVERNANCE_DEMO3_AGENDA_MODE=skip`** — agendas are not sent to Gemma. **Interrupt** the cell, re-run §1–§4 (pull latest repo), then §6. If you need agenda JSON: `GOVERNANCE_DEMO3_AGENDA_MODE=lite` (log must say `agenda lite` + prompt ~3k chars, not ~50k). |
| Demo 3 slow on **minutes** | **`GOVERNANCE_DEMO3_INPUT=auto`**: text-only when digital text exists. **Interrupt** past timeout. Reuse cache: `*.thinking.json` + `GOVERNANCE_FORCE_REPROCESS=0`. |
| Looks stuck on Gatekeeper API | AI Studio (1–3 min/file). Log shows `gemma START`. Local HF E2B uses Colab GPU. |
| Full model list spam | Keep `GOVERNANCE_LOG_LLM_CATALOG=0` |
| `count_triageable_files` missing | Runtime restart → re-run sections 1-3 |
| Gatekeeper OOM / CUDA | Only with **local HF** — use **L4 GPU + High RAM**; avoid `GOVERNANCE_GATEKEEPER_FORCE_HF=1` unless you need it. |
| Demo 3 empty `.thoughts.md` | Confirm `THINKING_MODEL` resolved to `gemma-4-31b-it`; set `GOVERNANCE_FORCE_THINKING=1` if needed |
| `429 RESOURCE_EXHAUSTED` / page 1 Gemma failed | Auto-retry is in `genai_quota_retry.py` (reload §3). Pace HIGH pages: `GOVERNANCE_GENAI_INTER_CALL_DELAY_HIGH_SECONDS=8`. More retries: `GOVERNANCE_GENAI_QUOTA_RETRIES=8`. Optional: `GOVERNANCE_GENAI_MODEL=gemma-4-31b-it` if listed on your key. |
| COFOG-01 on parks / unclear theme | Open `02_gemma_json/.../*.thinking.theme_audit.json` and `03_human_summaries/.../_meeting_summary.md` (theme table + flags). COFOG follows `primary_theme`; parks should be **Parks and Recreation → COFOG-08**. Re-run Demo 3 with `GOVERNANCE_FORCE_REPROCESS=1`. |
| No video summary / mermaid / one PDF only | Consolidated summary: `03_human_summaries/.../meetings/.../_meeting_summary.md` + `policy_drift.mmd`. Demo 4 + drift need audio in inventory (`audio≥1` in §5). Flat `*.mp4` at jurisdiction root are walked automatically; `_video_assets/*.opus` need the bytes on disk (`GOVERNANCE_INVENTORY_VIDEO_ASSETS=1`, default). Demo 4a for transcripts. |
| `2026_05_06/2026-05-06/` double date | Auto-repair hoists to `meetings/2026_05_06/session/` on organize + consolidated summary. Re-run §6; or set `GOVERNANCE_ORGANIZE_MEETINGS=1` after reload §3. |
| No `_meeting_summary.md` in meeting folder | Default: written beside agenda/minutes **and** under `03_human_summaries/…`. Look for `meetings/…/session/_meeting_summary.md` after repair. Set `GOVERNANCE_CONSOLIDATED_SUMMARY_IN_RAW=0` to only use processed tree. |
| Minutes / audio missing from summary | Demo 3 now runs **per meeting** (agenda + minutes). Demo 4 chunks + drift merged into summary when model accepts audio. Re-run with `GOVERNANCE_FORCE_REPROCESS=1`. |
| `.mp4` not in Demo 4 | Inventory lists video under `audio`; set `GOVERNANCE_DEMO4_VIDEO_CHUNKS=1` (default) for `video/mp4` segments |
| `reuse existing page JSON` / stale outputs | Set `GOVERNANCE_FORCE_REPROCESS=1` and re-run §2→§6, or delete the matching `02_gemma_json/.../page_*.json` / `.thinking.json` |
| Demo 3 `Thinking budget is not supported` | Fixed in code: Gemma gets `include_thoughts` without `thinking_budget`. Re-run §3+§6 with `GOVERNANCE_FORCE_REPROCESS=1` on failed PDFs. |
| Colab looks stuck (no new lines) | Normal on first HF load or each 15‑min chunk (3–15 min). You should see `… still HF weights` / `… still HF generate` every ~45s. Set `GOVERNANCE_HEARTBEAT_SECONDS=30` for faster pings; `0` to disable. |
| Demo 4 fails / no chunks | Need **GPU** + **HF_TOKEN** + `GOVERNANCE_DEMO4_USE_HF=1`. Logs: `ingest opus_15min` then `Hugging Face google/gemma-4-E2B-it`. Opus/ffmpeg reuse `_scratch_audio_chunks` on Drive. Set `GOVERNANCE_DEMO4_USE_HF=0` only to try Google API (Edge models often missing). |
| `no such file` … `2026-02-18/agenda/` | Stale paths after `session/` repair. Re-run **§6** (reload + remap). Files live under `meetings/2026_02_18/session/agenda/`. |
| `meeting brief LLM failed: NoneType ... models` | Reload §3 after pull; brief now builds its own GenAI client. |
| No data root | Sync pilots with `01_copy_scraped_meetings_cache_to_gdrive.py` |
| `Could not find c1_gemma_4_good` / `colab_paths.py` | **Colab:** re-run §1 (clones to `/content/c1_gemma_4_good`). **Cursor/local:** `os.environ['OPEN_NAVIGATOR_ROOT']='/path/to/c1_gemma_4_good'` then §1. Prefer **`run_in_colab.ipynb`** from this repo. |


### Drive outputs

```
03_processed_outputs/
  00_logs/triage_report_*.json         ← what was kept vs excluded
  02_gemma_json/                       ← Demo 1–4 JSON + sidecars
  03_human_summaries/                  ← Smart Brevity (if generated)
```

**Live demo tip:** open `00_logs/gatekeeper_*.log` on Drive during §6. Before API calls you should see walk/classify lines; during triage, `gemma START` / `gemma DONE`.
